# 🍺 Chopp & Cia · Inteligência de Risco em Comodato
# 🌲 Notebook 06 — Random Forest

**Projeto Integrador VI** · 2º Semestre/2026 · FATEC Votorantim

---

## 🎯 Responsabilidade única deste notebook

Treinar, avaliar e registrar **um algoritmo**: Random Forest. Nada mais.

| Entrada | Saída |
| :--- | :--- |
| `dataset_split_<SPLIT_HASH>` (notebook 03) | runs no MLflow + modelo registrado |

Este notebook **não** lê o banco, **não** faz EDA e **não** decide a partição. Cada uma
dessas responsabilidades tem seu próprio notebook, e a separação é o que permite tunar
Random Forest vinte vezes sem reprocessar o dump do ERP nem re-executar a análise
exploratória.

## 🧪 Como este notebook é usado

Ele executa **um lote de experimentos por vez**. Você declara no painel *qual pergunta*
o lote responde, e ele varre o eixo correspondente registrando cada variação como um run
do MLflow.

| `ESTUDO` | A pergunta |
| :--- | :--- |
| `hiperparametros` | qual configuração do algoritmo aprende melhor? |
| `grade_cruzada` | há interação entre dois parâmetros? |
| `elegibilidade` | quantos clientes vale a pena usar no treino? |
| `janela_temporal` | qual histórico generaliza melhor? |
| `ablacao` | quanto do desempenho vem do vazamento conhecido? |
| `baseline_unico` | rodada única de referência |

Trocar de estudo é trocar **uma linha** do painel. Cada lote vira um run pai no MLflow,
com um filho por variação.

## 🔗 Como comparar com os outros modelos

Os notebooks 04, 05 e 06 leem **a mesma tabela de split**. Enquanto o `SPLIT_HASH` for
o mesmo nos três, filtrar `tags.split_hash` no painel do MLflow e agrupar por
`tags.modelo` compara os algoritmos sobre exatamente as mesmas linhas — qualquer
diferença vem do algoritmo, não do sorteio.

## 📐 Por que Random Forest neste problema

Uma árvore isolada é instável — mudar poucas linhas do treino produz uma árvore
diferente. A floresta resolve isso pela média de muitas árvores, cada uma treinada num
*bootstrap* das linhas e vendo apenas um subconjunto aleatório das colunas em cada
divisão.

**A parte que importa é o `max_features='sqrt'`**, não o bootstrap. Se todas as árvores
vissem todas as colunas, escolheriam a mesma variável no topo e ficariam quase
idênticas — a média de árvores parecidas não reduz variância. Forçar cada divisão a
escolher entre poucas colunas é o que as torna genuinamente diferentes, e é daí que
vem o ganho.

| Ponto forte | Limitação |
| :--- | :--- |
| muito mais estável que uma árvore só | perde a leitura de regra explícita |
| captura interações sem declará-las | custo de treino e de inferência bem maior |
| importâncias mais confiáveis | importância por impureza favorece alta cardinalidade |
| raramente é o pior dos três | pode ser excessivo para amostra pequena |

### O eixo desta varredura: `max_depth`

Numa floresta, a profundidade importa **menos** do que numa árvore isolada: a média
entre árvores já reduz a variância que a poda combateria. `max_depth=None` (árvores
completas) é o padrão do sklearn justamente por isso, e frequentemente funciona bem.

O que observar: se a curva ficar plana a partir de certa profundidade, o valor menor é
preferível — mesma performance, menos custo e menos risco de decorar.

> **Antes de adotar a floresta**, compare com o notebook 05. Se a árvore isolada
> entregar MCC estatisticamente indistinguível, ela vence pela interpretabilidade —
> num contexto de crédito, poder explicar a decisão a um cliente tem valor próprio.

## 🎛️ Parâmetros (sem caminhos fixos)

Nenhum caminho de arquivo está escrito no código deste notebook. A origem dos
parâmetros depende de onde ele roda:

| Ambiente | Mecanismo | Onde aparece |
| :--- | :--- | :--- |
| **Databricks** | `dbutils.widgets` | campos no **topo** do notebook |
| **Local** (VS Code / Jupyter) | diálogo do sistema | janela de seleção de arquivo |


### Parâmetros deste notebook

| Parâmetro | O que é | Local | Databricks |
| :--- | :--- | :--- | :--- |
| `split_hash` | **qual partição usar** (o 03 imprime) | *lido do CSV* | campo de texto |
| `csv_split` | o CSV do notebook 03 | seletor de arquivo | *não usado* |
| `catalogo` · `schema` | onde a tabela vive | *não usado* | campo de texto |
| `estudo` | **a pergunta deste lote** | valor do painel | dropdown |
| `experiment_name` | experimento MLflow | campo | campo de texto |

Os dois que você mais vai mexer são o **split** (uma vez, no começo) e o **estudo**
(a cada lote).

> ⚠️ Os notebooks 04, 05 e 06 precisam apontar para o **mesmo split**. É isso que
> torna a comparação entre algoritmos pareada — hashes diferentes significam linhas
> diferentes, e a diferença de métrica passa a vir do sorteio, não do algoritmo.


### Como funciona localmente

Na primeira execução abre-se o diálogo do Windows. A escolha fica memorizada em
`~/.chopp_risco_params.json`, então **as execuções seguintes não perguntam nada** — o
diálogo só reaparece se o arquivo tiver sido movido, ou se você definir
`FORCAR_SELECAO = True`.

O diálogo é a **caixa nativa do Windows** (`comdlg32`), a mesma do Explorer — não o
`tkinter`. A diferença importa: o tkinter precisa criar uma janela-mãe para ancorar o
diálogo, e dentro do kernel do Jupyter essa janela nasce sem foco e atrás do editor.
A API nativa não cria janela nenhuma.

### 🔧 Se ainda assim o seletor não abrir

Em algumas instalações a janela simplesmente não aparece. Não é preciso lutar com
ela: preencha `CAMINHOS_MANUAIS` no topo da célula e o diálogo deixa de ser usado.

```python
CAMINHOS_MANUAIS = {
    "sql_file": r"C:\dados\DB_POWER_SYS.sql",
}
```

O `r` antes das aspas é necessário para que a barra invertida do Windows não seja
lida como caractere de escape. O que estiver preenchido ali tem **precedência sobre
tudo** — sobre o cache e sobre o diálogo.

### Como funciona no Databricks

Os campos aparecem no topo do notebook assim que a célula roda pela primeira vez.
Preencha e re-execute. Não há seletor de arquivo em notebook do Databricks — o caminho
do Volume é digitado, no formato `/Volumes/<catálogo>/<schema>/<volume>/arquivo`.



> **Sobre segurança:** tirar o caminho do código resolve **portabilidade** (o notebook
> roda na máquina de qualquer pessoa do grupo) e evita expor a estrutura de diretórios
> num repositório público. Não é um controle de acesso: quem executa o notebook lê o
> mesmo arquivo de qualquer forma. O controle de acesso real, no Databricks, vem das
> permissões do Unity Catalog sobre o Volume e as tabelas.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PARAMETRIZAÇÃO SEM CAMINHOS FIXOS
#
#  Nenhum caminho de arquivo é escrito no código. A origem dos parâmetros
#  depende de onde o notebook roda:
#
#    Databricks → dbutils.widgets, os campos que aparecem no topo do notebook.
#                 É o mecanismo nativo da plataforma; não existe file picker
#                 em notebook do Databricks.
#    Local      → caixa de diálogo NATIVA do Windows (a mesma do Explorer),
#                 com a escolha memorizada num JSON. Sem tkinter: ele cria uma
#                 janela-mãe que, dentro do kernel do Jupyter, nasce sem foco e
#                 atrás do VS Code — o diálogo abria, mas ficava invisível.
#
#  O cache local existe para que a segunda execução não reabra o diálogo: ele
#  só volta a aparecer se o arquivo tiver sumido ou se você pedir explicitamente
#  com FORCAR_SELECAO = True.
# ══════════════════════════════════════════════════════════════════════════════
import os
import json
from pathlib import Path

# Colocar em True reabre os diálogos mesmo havendo escolha memorizada.
# No Databricks não tem efeito: lá os widgets já são visíveis e editáveis.
FORCAR_SELECAO = True

# ── Caminhos informados à mão (alternativa ao diálogo) ────────────────────────
#
#  Se por qualquer motivo o seletor gráfico não abrir na sua máquina, preencha
#  aqui e o diálogo deixa de ser necessário. O que estiver preenchido TEM
#  PRECEDÊNCIA sobre tudo: sobre o cache e sobre o diálogo.
#
#  Deixe como está (strings vazias) para usar o seletor gráfico normalmente.
#
#  Use string "crua" (o r antes das aspas) para que a barra invertida do
#  Windows não seja interpretada como escape:
#      CAMINHOS_MANUAIS = {"sql_file": r"C:\dados\DB_POWER.sql"}
CAMINHOS_MANUAIS = {
    # "sql_file":         r"",
    # "output_dir":       r"",
    # "csv_consolidado":  r"",
    # "csv_split":        r"",
}

try:
    dbutils                                     # type: ignore # noqa: F821
    NO_DATABRICKS = True
except NameError:
    NO_DATABRICKS = False

# Onde a escolha local fica memorizada. Fica ao lado do notebook, e não no
# diretório de dados: é preferência de máquina, não dado do projeto.
_CACHE_PARAMS = Path.home() / ".chopp_risco_params.json"


def _cache_ler() -> dict:
    """Lê as escolhas memorizadas. Cache corrompido não pode derrubar o notebook."""
    try:
        if _CACHE_PARAMS.exists():
            return json.loads(_CACHE_PARAMS.read_text(encoding="utf-8"))
    except Exception:
        pass
    return {}


def _cache_gravar(chave: str, valor: str) -> None:
    """Memoriza uma escolha. Falha de escrita é irrelevante — só perde o atalho."""
    try:
        d = _cache_ler()
        d[chave] = str(valor)
        _CACHE_PARAMS.write_text(
            json.dumps(d, indent=2, ensure_ascii=False), encoding="utf-8"
        )
    except Exception:
        pass


# ══════════════════════════════════════════════════════════════════════════════
#  DIÁLOGO NATIVO DO WINDOWS (ctypes → comdlg32)
#
#  Por que não tkinter: o tkinter cria uma janela-mãe (Tk) para ancorar o
#  diálogo, e dentro do kernel do Jupyter essa janela nasce sem foco e atrás do
#  VS Code. O diálogo abre — mas fica invisível, e a célula parece travada.
#
#  A API abaixo é a mesma que o Explorer e qualquer programa Windows usam.
#  Não cria janela nenhuma: não há o que ficar atrás de nada.
# ══════════════════════════════════════════════════════════════════════════════
def _dialogo_windows_arquivo(titulo: str, filtros, inicial=None):
    """
    Caixa de seleção de arquivo nativa (GetOpenFileNameW). Caminho ou None.

    filtros: lista [(rótulo, padrão), ...] — ex.: [("Dump SQL", "*.sql")]
    """
    import ctypes
    from ctypes import wintypes

    class OPENFILENAMEW(ctypes.Structure):
        _fields_ = [
            ("lStructSize", wintypes.DWORD), ("hwndOwner", wintypes.HWND),
            ("hInstance", wintypes.HINSTANCE), ("lpstrFilter", wintypes.LPCWSTR),
            ("lpstrCustomFilter", wintypes.LPWSTR), ("nMaxCustFilter", wintypes.DWORD),
            ("nFilterIndex", wintypes.DWORD), ("lpstrFile", wintypes.LPWSTR),
            ("nMaxFile", wintypes.DWORD), ("lpstrFileTitle", wintypes.LPWSTR),
            ("nMaxFileTitle", wintypes.DWORD), ("lpstrInitialDir", wintypes.LPCWSTR),
            ("lpstrTitle", wintypes.LPCWSTR), ("Flags", wintypes.DWORD),
            ("nFileOffset", wintypes.WORD), ("nFileExtension", wintypes.WORD),
            ("lpstrDefExt", wintypes.LPCWSTR), ("lCustData", wintypes.LPARAM),
            ("lpfnHook", wintypes.LPVOID), ("lpTemplateName", wintypes.LPCWSTR),
            ("pvReserved", wintypes.LPVOID), ("dwReserved", wintypes.DWORD),
            ("FlagsEx", wintypes.DWORD),
        ]

    buf = ctypes.create_unicode_buffer(4096)

    # A API espera pares "rótulo\0padrão\0", terminados por um \0 extra.
    partes = []
    for rotulo, padrao in filtros:
        partes += [rotulo, padrao]
    filtro_api = "\0".join(partes) + "\0\0"

    ofn = OPENFILENAMEW()
    ofn.lStructSize = ctypes.sizeof(OPENFILENAMEW)
    ofn.hwndOwner = ctypes.windll.user32.GetForegroundWindow()
    ofn.lpstrFilter = filtro_api
    ofn.lpstrFile = ctypes.cast(buf, wintypes.LPWSTR)
    ofn.nMaxFile = 4096
    ofn.lpstrTitle = titulo
    ofn.lpstrInitialDir = inicial
    # NOCHANGEDIR: sem isto o diálogo muda o diretório de trabalho do kernel,
    # e caminhos relativos usados depois passam a apontar para outro lugar.
    ofn.Flags = 0x00001000 | 0x00000800 | 0x00000008 | 0x00080000

    if ctypes.windll.comdlg32.GetOpenFileNameW(ctypes.byref(ofn)):
        return buf.value or None
    return None


def _dialogo_windows_pasta(titulo: str):
    """Caixa de seleção de pasta nativa (SHBrowseForFolderW). Caminho ou None."""
    import ctypes
    from ctypes import wintypes

    class BROWSEINFOW(ctypes.Structure):
        _fields_ = [
            ("hwndOwner", wintypes.HWND), ("pidlRoot", ctypes.c_void_p),
            ("pszDisplayName", wintypes.LPWSTR), ("lpszTitle", wintypes.LPCWSTR),
            ("ulFlags", wintypes.UINT), ("lpfn", wintypes.LPVOID),
            ("lParam", wintypes.LPARAM), ("iImage", ctypes.c_int),
        ]

    shell32 = ctypes.windll.shell32
    nome = ctypes.create_unicode_buffer(4096)

    bi = BROWSEINFOW()
    bi.hwndOwner = ctypes.windll.user32.GetForegroundWindow()
    bi.pszDisplayName = ctypes.cast(nome, wintypes.LPWSTR)
    bi.lpszTitle = titulo
    bi.ulFlags = 0x00000001 | 0x00000040     # só diretórios + diálogo moderno

    shell32.SHBrowseForFolderW.restype = ctypes.c_void_p
    pidl = shell32.SHBrowseForFolderW(ctypes.byref(bi))
    if not pidl:
        return None
    try:
        caminho = ctypes.create_unicode_buffer(4096)
        shell32.SHGetPathFromIDListW.argtypes = [ctypes.c_void_p, wintypes.LPWSTR]
        ok = shell32.SHGetPathFromIDListW(pidl, caminho)
        return caminho.value if ok else None
    finally:
        # A lista de IDs é alocada pelo shell; liberá-la é responsabilidade nossa.
        ctypes.windll.ole32.CoTaskMemFree(ctypes.c_void_p(pidl))


def _dialogo_tkinter_arquivo(titulo: str, tipos, inicial=None):
    """Alternativa por tkinter, para quando a API do Windows não estiver disponível."""
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.update_idletasks()
    root.attributes("-topmost", True)
    root.lift()
    try:
        root.focus_force()
    except Exception:
        pass
    root.update()
    try:
        return filedialog.askopenfilename(
            parent=root, title=titulo, filetypes=tipos,
            initialdir=inicial or str(Path.home()),
        ) or None
    finally:
        try:
            root.destroy()
        except Exception:
            pass


def _dialogo_tkinter_pasta(titulo: str, inicial=None):
    """Alternativa por tkinter para seleção de pasta."""
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.update_idletasks()
    root.attributes("-topmost", True)
    root.lift()
    try:
        root.focus_force()
    except Exception:
        pass
    root.update()
    try:
        return filedialog.askdirectory(
            parent=root, title=titulo, initialdir=inicial or str(Path.home())
        ) or None
    finally:
        try:
            root.destroy()
        except Exception:
            pass


def _selecionar_arquivo(titulo: str, tipos, inicial=None):
    """
    Abre a caixa de seleção de arquivo. Retorna o caminho ou None.

    Tenta primeiro a API nativa do Windows (sem janela intermediária) e recorre
    ao tkinter fora do Windows ou se a API falhar.
    """
    if os.name == "nt":
        try:
            return _dialogo_windows_arquivo(titulo, tipos, inicial)
        except Exception as e:
            print(f"   ⚠️  Diálogo nativo falhou ({type(e).__name__}: {e});"
                  f" tentando tkinter...")

    try:
        return _dialogo_tkinter_arquivo(titulo, tipos, inicial)
    except ImportError:
        print("   ⚠️  Nenhum seletor disponível (sem tkinter).")
        return None
    except Exception as e:
        print(f"   ⚠️  Seletor indisponível ({type(e).__name__}: {e}).")
        return None


def _selecionar_pasta(titulo: str, inicial=None):
    """Abre a caixa de seleção de pasta. Retorna o caminho ou None."""
    if os.name == "nt":
        try:
            return _dialogo_windows_pasta(titulo)
        except Exception as e:
            print(f"   ⚠️  Diálogo nativo falhou ({type(e).__name__}: {e});"
                  f" tentando tkinter...")

    try:
        return _dialogo_tkinter_pasta(titulo, inicial)
    except ImportError:
        print("   ⚠️  Nenhum seletor disponível (sem tkinter).")
        return None
    except Exception as e:
        print(f"   ⚠️  Seletor indisponível ({type(e).__name__}: {e}).")
        return None


def widget(nome: str, padrao: str = "", rotulo: str = None,
           opcoes: list = None) -> str:
    """
    Declara um parâmetro de texto (ou dropdown) e devolve seu valor.

    No Databricks vira um campo no topo do notebook. Localmente devolve o valor
    memorizado, ou o padrão.

    A recriação do widget a cada execução é intencional: mudar o padrão no
    código passa a valer sem precisar remover o widget à mão.
    """
    rotulo = rotulo or nome
    if NO_DATABRICKS:
        try:
            if opcoes:
                dbutils.widgets.dropdown(nome, padrao or opcoes[0],
                                         opcoes, rotulo)          # noqa: F821
            else:
                dbutils.widgets.text(nome, padrao, rotulo)        # noqa: F821
        except Exception:
            pass   # widget já existe com outro tipo — o valor abaixo ainda serve
        try:
            return dbutils.widgets.get(nome)                      # noqa: F821
        except Exception:
            return padrao
    return _cache_ler().get(nome, padrao)


def parametro_arquivo(nome: str, titulo: str, tipos, padrao_databricks: str = "",
                      rotulo: str = None) -> str:
    """
    Resolve o caminho de um ARQUIVO de entrada.

    Databricks → widget de texto (caminho do Volume; não há file picker lá).
    Local      → escolha memorizada, ou diálogo do sistema.

    Falha com mensagem clara se nada for escolhido: seguir com caminho vazio
    produziria um FileNotFoundError muito adiante, sem contexto.
    """
    if NO_DATABRICKS:
        valor = widget(nome, padrao_databricks, rotulo or titulo)
        if not valor:
            raise ValueError(
                f"\n\n  ❌ Parâmetro '{nome}' vazio.\n\n"
                f"     Preencha o campo '{rotulo or titulo}' no topo do notebook\n"
                f"     com o caminho do arquivo no Volume, por exemplo:\n"
                f"       /Volumes/<catalogo>/<schema>/<volume>/arquivo.sql\n"
            )
        return valor

    # 1. Caminho informado à mão vence tudo — é a saída para quando o diálogo
    #    gráfico não abre.
    manual = (CAMINHOS_MANUAIS.get(nome) or "").strip()
    if manual:
        if not os.path.exists(manual):
            raise FileNotFoundError(
                f"\n\n  ❌ CAMINHOS_MANUAIS['{nome}'] aponta para um arquivo que\n"
                f"     não existe:\n\n       {manual}\n\n"
                f"     Corrija o caminho no topo desta célula.\n"
            )
        print(f"   ✍️  {nome}: caminho informado em CAMINHOS_MANUAIS")
        print(f"      {manual}")
        return manual

    # 2. Escolha memorizada de uma execução anterior
    memorizado = _cache_ler().get(nome)
    if memorizado and os.path.exists(memorizado) and not FORCAR_SELECAO:
        print(f"   📎 {nome}: usando a escolha memorizada")
        print(f"      {memorizado}")
        print(f"      (para escolher outro, defina FORCAR_SELECAO = True acima)")
        return memorizado

    if memorizado and not os.path.exists(memorizado):
        print(f"   ⚠️  O arquivo memorizado não existe mais:")
        print(f"      {memorizado}")

    # 3. Diálogo gráfico
    print(f"   📂 Abrindo o seletor de arquivo...")
    print(f"      ⚠️  A janela pode abrir ATRÁS do editor — procure na barra de tarefas.")
    escolhido = _selecionar_arquivo(
        titulo, tipos,
        inicial=os.path.dirname(memorizado) if memorizado else None,
    )
    if not escolhido:
        raise ValueError(
            f"\n\n  ❌ Nenhum arquivo selecionado para '{nome}'.\n\n"
            f"     Isso acontece se você cancelou o diálogo — ou se ele não chegou\n"
            f"     a aparecer (em algumas instalações do VS Code a janela do\n"
            f"     tkinter nasce sem foco).\n\n"
            f"     DUAS SAÍDAS:\n\n"
            f"     (a) informe o caminho à mão — preencha no topo desta célula:\n"
            f"           CAMINHOS_MANUAIS = {{\n"
            f"               \"{nome}\": r\"C:\\caminho\\para\\o\\arquivo\",\n"
            f"           }}\n"
            f"         e re-execute. É a opção que não depende de janela nenhuma.\n\n"
            f"     (b) re-execute a célula e procure a janela na barra de tarefas\n"
            f"         (ícone do Python) antes de clicar em qualquer outro lugar.\n"
        )
    _cache_gravar(nome, escolhido)
    print(f"   ✅ Selecionado e memorizado para as próximas execuções.")
    return escolhido


def parametro_pasta(nome: str, titulo: str, padrao_databricks: str = "",
                    rotulo: str = None) -> str:
    """
    Resolve o caminho de um DIRETÓRIO de saída.

    Diferença em relação a parametro_arquivo(): um diretório inexistente é
    criado em vez de recusado — é saída, não entrada.
    """
    if NO_DATABRICKS:
        valor = widget(nome, padrao_databricks, rotulo or titulo)
        if not valor:
            raise ValueError(
                f"\n\n  ❌ Parâmetro '{nome}' vazio.\n"
                f"     Preencha o campo '{rotulo or titulo}' no topo do notebook.\n"
            )
        return valor

    # 1. Caminho informado à mão vence tudo
    manual = (CAMINHOS_MANUAIS.get(nome) or "").strip()
    if manual:
        os.makedirs(manual, exist_ok=True)
        print(f"   ✍️  {nome}: caminho informado em CAMINHOS_MANUAIS")
        print(f"      {manual}")
        return manual

    # 2. Escolha memorizada
    memorizado = _cache_ler().get(nome)
    if memorizado and not FORCAR_SELECAO:
        os.makedirs(memorizado, exist_ok=True)
        print(f"   📎 {nome}: usando a escolha memorizada")
        print(f"      {memorizado}")
        return memorizado

    # 3. Diálogo gráfico
    print(f"   📂 Abrindo o seletor de pasta...")
    print(f"      ⚠️  A janela pode abrir ATRÁS do editor — procure na barra de tarefas.")
    escolhido = _selecionar_pasta(titulo, inicial=memorizado)
    if not escolhido:
        raise ValueError(
            f"\n\n  ❌ Nenhuma pasta selecionada para '{nome}'.\n\n"
            f"     Isso acontece se você cancelou o diálogo — ou se ele não chegou\n"
            f"     a aparecer (em algumas instalações do VS Code a janela do\n"
            f"     tkinter nasce sem foco).\n\n"
            f"     DUAS SAÍDAS:\n\n"
            f"     (a) informe o caminho à mão — preencha no topo desta célula:\n"
            f"           CAMINHOS_MANUAIS = {{\n"
            f"               \"{nome}\": r\"C:\\caminho\\para\\a\\pasta\",\n"
            f"           }}\n"
            f"         e re-execute. É a opção que não depende de janela nenhuma.\n\n"
            f"     (b) re-execute a célula e procure a janela na barra de tarefas.\n"
        )
    os.makedirs(escolhido, exist_ok=True)
    _cache_gravar(nome, escolhido)
    print(f"   ✅ Selecionada e memorizada para as próximas execuções.")
    return escolhido


print("✅ Parametrização carregada")
print(f"   ├─ Ambiente : {'Databricks (widgets)' if NO_DATABRICKS else 'Local (seletor de arquivo)'}")
if not NO_DATABRICKS:
    print(f"   └─ Cache    : {_CACHE_PARAMS}")
else:
    print(f"   └─ Os parâmetros aparecem como campos no TOPO do notebook.")

# A seleção local acontece nesta própria célula.
CSV_SPLIT = None if NO_DATABRICKS else parametro_arquivo(
    nome="csv_split",
    titulo="Selecione o dataset de split (CSV do notebook 03)",
    tipos=[("CSV", "*.csv"), ("Todos os arquivos", "*.*")],
)

## 🎛️ 1. Painel de Controle

Toda variável capaz de mudar o resultado mora nesta célula. As seguintes apenas leem
daqui — é a condição para que o registro no MLflow seja fiel, porque um hiperparâmetro
escrito no meio do notebook é um hiperparâmetro que o log não enxerga.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  BIBLIOTECAS
# ══════════════════════════════════════════════════════════════════════════════
import os
import json
import hashlib
import pickle
import warnings
import itertools
from datetime import datetime
from typing import Dict, Any, List, cast

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
from IPython.display import display

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, brier_score_loss, fbeta_score,
)
from sklearn.ensemble import RandomForestClassifier

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

from sklearn import set_config
set_config(display="diagram")
sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})

try:
    spark                                       # type: ignore # noqa: F821
    NO_DATABRICKS = True
except NameError:
    NO_DATABRICKS = False

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PAINEL DE CONTROLE DO EXPERIMENTO
#
#  Toda variável capaz de mudar o resultado mora AQUI. As células seguintes só
#  LEEM deste bloco — um hiperparâmetro escrito no meio do notebook é um
#  hiperparâmetro que o log do MLflow não enxerga.
# ══════════════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 1 · ENTRADA — de onde vêm os dados
# ══════════════════════════════════════════════════════════════════════════════

# ⬇️ Split publicado pelo notebook 03. O sufixo é o SPLIT_HASH, que o 03
#    imprime ao terminar.
#    Os notebooks 04, 05 e 06 devem apontar para o MESMO split — é isso que
#    torna a comparação entre algoritmos pareada.
#
#    Databricks → preencha 'split_hash' no campo do topo do notebook.
#    Local      → o diálogo de seleção abre para escolher o CSV do split, e o
#                 hash é lido do próprio arquivo.
_CATALOGO = widget("catalogo", "projetointegrador", "Catálogo (Unity Catalog)")
_SCHEMA = widget("schema", "projetointegrador", "Schema")

if NO_DATABRICKS:
    _SPLIT_HASH_PARAM = widget("split_hash", "", "SPLIT_HASH (impresso pelo notebook 03)")
    if not _SPLIT_HASH_PARAM:
        raise ValueError(
            "\n\n  Campo 'SPLIT_HASH' vazio.\n\n"
            "     Preencha-o no topo do notebook com o hash que o notebook 03\n"
            "     imprime ao terminar, e re-execute esta celula.\n\n"
            "     Sem ele nao ha como saber qual particao usar - e apontar para\n"
            "     o split errado invalidaria a comparacao entre os modelos.\n"
        )
    TABELA_SPLIT = f"{_CATALOGO}.{_SCHEMA}.dataset_split_{_SPLIT_HASH_PARAM}"
else:
    TABELA_SPLIT = None

EXPERIMENT_NAME = widget(
    "experiment_name",
    "/Users/brunofsaraujo@hotmail.com/Chopp_Cia_Experimentos",
    "Experimento MLflow",
)
NOME_REGISTRADO = widget("modelo_registrado", "workspace.default.chopp_risco_random_forest", "Nome no Model Registry")

# Semente única, propagada a CV e estimadores. Não é hiperparâmetro: é
# dispositivo de REPRODUTIBILIDADE. Variá-la para "melhorar" a métrica é
# escolher o ruído favorável, não um modelo melhor.
RANDOM_STATE = 42


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 2 · O ESTUDO — a pergunta que este lote responde
#
#  ⬇️ ESTE É O PARÂMETRO QUE VOCÊ MAIS VAI MEXER ⬇️
#
#  Opções e o que cada uma responde:
#
#    'hiperparametros'  varre UM parâmetro do algoritmo
#                       → qual configuração aprende melhor?
#
#    'grade_cruzada'    produto cartesiano de dois ou mais parâmetros
#                       → há interação entre eles?
#                       ⚠️ com amostra pequena, varrer muitas combinações
#                          otimiza ruído: o resultado parece melhor sem ser
#
#    'elegibilidade'    varre min_compras filtrando o split congelado
#                       → quantos clientes vale a pena usar?
#
#    'janela_temporal'  varre a recência máxima
#                       → qual histórico generaliza melhor?
#
#    'ablacao'          remove conjuntos de features
#                       → quanto do desempenho vem do vazamento conhecido?
#
#    'baseline_unico'   uma configuração só
#                       → rodada de referência
# ══════════════════════════════════════════════════════════════════════════════
ESTUDO = widget(
    "estudo", "hiperparametros", "Estudo (a pergunta deste lote)",
    opcoes=["hiperparametros", "grade_cruzada", "elegibilidade",
            "janela_temporal", "ablacao", "baseline_unico"],
)

NOTA_ESTUDO = (
    "Varredura de max_depth com n_estimators fixo. Numa floresta, a profundidade importa menos que numa arvore isolada — a media entre arvores ja reduz variancia — mas continua governando o custo."
)


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 3 · GRADE DO ESTUDO — o que varre, dentro do estudo escolhido
#  Cada estudo lê a chave que lhe corresponde; as demais ficam inertes.
# ══════════════════════════════════════════════════════════════════════════════
GRADES = {

    # ── Para ESTUDO = 'hiperparametros' ──────────────────────────────────────
    "hiperparametros": {
    "parametro": "max_depth",
    "valores": [3, 4, 6, 8, 12, None],
},

    # ── Para ESTUDO = 'grade_cruzada' ────────────────────────────────────────
    "grade_cruzada": {
        "parametros": {
            "max_depth": [4, 8, None],
            "n_estimators": [100, 300],
        },
    },

    # ── Para ESTUDO = 'elegibilidade' ────────────────────────────────────────
    # Mesma grade da seção 3.7 do notebook 02, para que a leitura estrutural de
    # lá e a leitura de performance daqui sejam diretamente comparáveis.
    "elegibilidade": {
        "valores": [0, 1, 2, 3, 5, 8, 12],
    },

    # ── Para ESTUDO = 'janela_temporal' ──────────────────────────────────────
    # Mantém quem comprou nos últimos N dias. Responde se treinar só na carteira
    # recente generaliza melhor do que usar a base histórica inteira.
    "janela_temporal": {
        "valores": [90, 180, 365, 730, 99999],   # 99999 = sem recorte
    },

    # ── Para ESTUDO = 'ablacao' ──────────────────────────────────────────────
    # O experimento mais importante deste projeto. MEDIA_DIAS_ATRASO_* compartilha
    # origem aritmética com TAXA_ATRASO_*, que constrói o alvo. Este estudo mede
    # exatamente quanto do desempenho depende dessas features — e portanto quanto
    # do resultado é capacidade preditiva genuína.
    "ablacao": {
        "conjuntos": [
            {"nome": "completo (todas as features)", "remover": []},
            {"nome": "sem MEDIA_DIAS_ATRASO_PAG", "remover": ["MEDIA_DIAS_ATRASO_PAG"]},
            {"nome": "sem MEDIA_DIAS_ATRASO_COM", "remover": ["MEDIA_DIAS_ATRASO_COM"]},
            {"nome": "sem ambas as MEDIA_DIAS_ATRASO",
              "remover": ["MEDIA_DIAS_ATRASO_PAG", "MEDIA_DIAS_ATRASO_COM"]},
            {"nome": "só RFM (sem nada de atraso)",
              "remover": ["MEDIA_DIAS_ATRASO_PAG", "MEDIA_DIAS_ATRASO_COM",
                          "TOTAL_PARCELAS", "TOTAL_COMODATOS"]},
        ],
    },

    # ── Para ESTUDO = 'baseline_unico' ───────────────────────────────────────
    "baseline_unico": {},
}

GRADE_ESTUDO = GRADES.get(ESTUDO, {})


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 4 · HIPERPARÂMETROS-BASE DO MODELO
#  Ponto de partida do lote. O estudo sobrescreve o parâmetro que varre; os
#  demais permanecem como declarados aqui.
# ══════════════════════════════════════════════════════════════════════════════
CLASSE_MODELO = RandomForestClassifier
NOME_MODELO = "Random Forest"
SLUG_MODELO = "random_forest"
COR_MODELO = "#E74C3C"

HP_BASE: Dict[str, Any] = {
    "n_estimators": 200,         # mais árvores = mais estável, custo linear
    "max_depth": 8,
    "min_samples_split": 2,
    "min_samples_leaf": 3,       # ↑ suaviza as folhas e reduz variância
    "max_features": "sqrt",      # descorrelaciona as árvores — a ideia central
    "bootstrap": True,
    "class_weight": "balanced",
    # n_jobs=1: no serverless a contagem de cores é baixa e não configurável;
    # -1 não escala e ainda disputa CPU com o paralelismo da CV.
    "n_jobs": 1,
}


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 5 · PRÉ-PROCESSAMENTO
# ══════════════════════════════════════════════════════════════════════════════
PREPROC: Dict[str, Any] = {
    "scaler_numerico": "none",       # standard | minmax | robust | none
    "encoder_categorico": "onehot",
    "onehot_drop": None,        # floresta não precisa de escala nem sofre com colinearidade
    "onehot_handle_unknown": "ignore",   # categoria nova em produção não quebra
    "remainder": "drop",
}


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 6 · VALIDAÇÃO CRUZADA
# ══════════════════════════════════════════════════════════════════════════════
CV: Dict[str, Any] = {
    "n_splits": 5,
    "shuffle": True,
    # n_jobs=None (sequencial): o serverless não oferece temp dir gravável para o
    # loky, e com poucas centenas de linhas o paralelismo não paga seu overhead.
    "n_jobs": None,
    "return_train_score": True,   # habilita o gap treino−validação
}

# Métricas coletadas na CV. A classe positiva é MAIORIA: por isso o conjunto
# inclui métricas insensíveis a prevalência (balanced_accuracy, MCC) além das
# tradicionais.
SCORING_CV: Dict[str, str] = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
    "balanced_accuracy": "balanced_accuracy",
    "matthews_corrcoef": "matthews_corrcoef",
}


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 7 · SELEÇÃO DO CAMPEÃO
#
#  ⚠️ SOBRE 'origem' — leia antes da segunda rodada:
#     'teste' = escolhe pelo holdout. Válido para UMA rodada. A cada configuração
#               adicional comparada pelo test_*, o holdout deixa de ser estimativa
#               não-viesada — com poucos negativos, o "melhor test MCC" após
#               várias rodadas é ruído escolhido.
#     'cv'    = escolhe pela validação cruzada do treino, deixando o holdout
#               intocado para uma única medição final. É a opção CORRETA quando
#               se compara muitas configurações — que é exatamente o caso de um
#               lote com 7 variações.
# ══════════════════════════════════════════════════════════════════════════════
SELECAO: Dict[str, Any] = {
    # MCC, não AUC: só sobe quando AMBAS as classes são bem classificadas.
    # O baseline majoritário tem MCC = 0 por construção — um piso claro.
    "metrica": "MCC",
    "origem": "cv",              # cv | teste  (veja o aviso acima)
    "maior_melhor": True,
    "criterio_desempate": "Balanced_Accuracy",
}

INTERVALO_CONFIANCA = {"z": 1.96, "nivel": 0.95}

# Sem esta âncora nenhuma métrica significa nada.
BASELINE = {"ativo": True, "strategy": "most_frequent"}

THRESHOLD: Dict[str, Any] = {
    "classificacao": 0.50,   # ≥ ⇒ classificado como alto risco
    "faixa_baixo": 0.35,     # prob < 0.35        ⇒ BAIXO
    "faixa_medio": 0.65,     # 0.35 ≤ prob < 0.65 ⇒ MÉDIO; ≥ 0.65 ⇒ ALTO
    # Grade varrida out-of-fold para escolher o ponto de operação por curva
    "sweep_inicio": 0.20,
    "sweep_fim": 0.80,
    "sweep_passo": 0.05,
}


# ══════════════════════════════════════════════════════════════════════════════
#  BLOCO 8 · METAS DE NEGÓCIO
#  Fixadas antes do experimento. Não variam entre rodadas de tuning — mover a
#  trave depois de ver o resultado não é tuning.
# ══════════════════════════════════════════════════════════════════════════════
METAS_KPI: Dict[str, float] = {
    "auc_roc": 0.75,
    "recall_classe1": 0.70,
    "accuracy": 0.65,
}


# ─── Validação defensiva do painel ────────────────────────────────────────────
# Erro de digitação em hiperparâmetro é silencioso e caro: o notebook roda até o
# fim e você só descobre depois que a rodada mediu outra coisa.
def validar_painel() -> list:
    """Checa coerência interna do painel. Retorna a lista de problemas."""
    p = []
    if not 0 < THRESHOLD["classificacao"] < 1:
        p.append("THRESHOLD['classificacao'] deve estar em (0, 1).")
    if not THRESHOLD["faixa_baixo"] < THRESHOLD["faixa_medio"]:
        p.append("THRESHOLD['faixa_baixo'] deve ser menor que 'faixa_medio'.")
    if not THRESHOLD["sweep_inicio"] < THRESHOLD["sweep_fim"]:
        p.append("THRESHOLD['sweep_inicio'] deve ser menor que 'sweep_fim'.")
    if THRESHOLD["sweep_passo"] <= 0:
        p.append("THRESHOLD['sweep_passo'] deve ser positivo.")
    if CV["n_splits"] < 2:
        p.append("CV['n_splits'] deve ser >= 2.")
    if SELECAO["origem"] not in {"teste", "cv"}:
        p.append("SELECAO['origem'] deve ser 'teste' ou 'cv'.")
    if ESTUDO not in GRADES:
        p.append(f"ESTUDO='{ESTUDO}' não tem grade declarada em GRADES.")
    if NO_DATABRICKS and not TABELA_SPLIT:
        p.append("TABELA_SPLIT não resolvida — preencha o campo SPLIT_HASH no topo.")
    if not NO_DATABRICKS and not CSV_SPLIT:
        p.append("CSV_SPLIT não resolvido — selecione o arquivo de split.")
    return p


_PROBLEMAS = validar_painel()

print("=" * 78)
print(f"{'PAINEL DE CONTROLE · ' + NOME_MODELO:^78}")
print("=" * 78)
print(f"  Estudo       : {ESTUDO}")
print(f"  Modelo       : {CLASSE_MODELO.__name__}")
print(f"  Entrada      : {(TABELA_SPLIT or CSV_SPLIT or '?').split(chr(92))[-1].split('.')[-1] if not NO_DATABRICKS else TABELA_SPLIT.split('.')[-1]}")
print("-" * 78)
print(f"  HP base      : {', '.join(f'{k}={v}' for k, v in HP_BASE.items())}")
print(f"  Pré-proc     : {PREPROC['scaler_numerico']} + onehot(drop={PREPROC['onehot_drop']})")
print(f"  Validação    : {CV['n_splits']}-Fold estratificado")
print(f"  Campeão por  : {SELECAO['metrica']} ({SELECAO['origem']}), "
      f"desempate {SELECAO['criterio_desempate']}")
print(f"  Threshold    : {THRESHOLD['classificacao']:.2f}")
print("-" * 78)

if _PROBLEMAS:
    print("  ❌ PAINEL INCONSISTENTE — corrija antes de treinar:")
    for _p in _PROBLEMAS:
        print(f"     • {_p}")
    raise ValueError(f"{len(_PROBLEMAS)} problema(s) no painel de controle.")

print(f"  ✅ Painel validado")
if SELECAO["origem"] == "teste":
    print()
    print("  ⚠️  SELECAO['origem']='teste': o campeão sai do holdout. Válido para uma")
    print("      rodada isolada; num lote com várias variações, o holdout passa a")
    print("      fazer parte da escolha e deixa de estimar generalização. Prefira 'cv'.")
print("=" * 78)

## 📂 2. Carga do Split Congelado

O notebook 03 já decidiu quais linhas treinam e quais testam. Aqui apenas **lemos**
essa decisão — não há `train_test_split` neste notebook, e isso é deliberado.

Se cada notebook de modelo refizesse o sorteio, bastaria uma linha a mais na origem
para as partições divergirem, e a comparação entre Regressão, Árvore e Random Forest
deixaria de ser pareada sem que nada avisasse.

> A célula reporta `SPLIT_HASH` e `DATA_VERSION` lidos da tabela. **Confira que são os
> mesmos nos três notebooks** antes de comparar resultados.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CARGA DO SPLIT CONGELADO
#  Nenhum train_test_split aqui: a partição foi decidida no notebook 03 e é lida
#  como está. É o que garante que os três modelos vejam as MESMAS linhas.
# ══════════════════════════════════════════════════════════════════════════════
_origem = TABELA_SPLIT if NO_DATABRICKS else os.path.basename(CSV_SPLIT)
print(f"📂 Carregando {_origem}...\n")

if NO_DATABRICKS:
    df_split = spark.read.table(TABELA_SPLIT).toPandas()             # noqa: F821
else:
    df_split = pd.read_csv(CSV_SPLIT, sep=";", encoding="utf-8-sig", low_memory=False)

if "_SPLIT" not in df_split.columns:
    raise KeyError(
        "Coluna _SPLIT ausente — esta tabela não parece ter vindo do notebook 03. "
        "Verifique TABELA_SPLIT."
    )

# ── Proveniência ──────────────────────────────────────────────────────────────
SPLIT_HASH = str(df_split["_SPLIT_HASH"].iloc[0]) if "_SPLIT_HASH" in df_split.columns else "desconhecido"
DATA_VERSION = str(df_split["_DATA_VERSION"].iloc[0]) if "_DATA_VERSION" in df_split.columns else "desconhecida"

# ── Recuperação das features ──────────────────────────────────────────────────
# As features são o que a tabela traz, menos alvo, identidade e metadados. Deduzir
# em vez de redeclarar evita a divergência silenciosa entre o que o notebook 03
# materializou e o que este notebook acha que materializou.
_META = {"_SPLIT", "_SPLIT_HASH", "_DATA_VERSION"}
TARGET = "ALTO_RISCO"

# Identificação: a chave, mais qualquer coluna que identifique a PESSOA.
# Por decisão de projeto o pipeline não carrega nome nem fantasia, mas a exclusão é por
# PADRÃO e não por lista fixa — assim, se uma carga futura reintroduzir uma
# coluna nominal, ela é barrada aqui em vez de virar feature em silêncio.
#
# Por que isso importa: nome não prevê inadimplência. Um modelo com acesso a
# ele aprende CLIENTES em vez de COMPORTAMENTO, e passa a discriminar por
# identidade onde deveria discriminar por conduta.
_PADROES_NOMINAIS = ("NOME", "NM_", "FANTASIA", "RAZAO", "RAZÃO",
                     "CPF", "CNPJ", "EMAIL", "TELEFONE", "ENDERECO", "ENDEREÇO")
_IDENTIDADE = {"ID_PESSOA"} | {
    c for c in df_split.columns
    if any(p in c.upper() for p in _PADROES_NOMINAIS)
}

if TARGET not in df_split.columns:
    raise KeyError(f"Alvo '{TARGET}' ausente na tabela de split.")

_nominais = sorted(_IDENTIDADE - {"ID_PESSOA"})
if _nominais:
    print(f"   ⚠️  Colunas de identificação nominal na tabela de split: {_nominais}")
    print(f"       Excluídas das features — por decisão de projeto não deveriam existir.")
    print(f"       Verifique COLUNAS_NECESSARIAS no notebook 01.\n")

_todas = [c for c in df_split.columns if c not in _META | _IDENTIDADE | {TARGET}]

# Numéricas vs. categóricas pelo dtype, com conversão defensiva: o CSV devolve
# tudo como string e o OneHotEncoder aceitaria uma coluna numérica como categoria.
FEATURES_CAT = [c for c in _todas if df_split[c].dtype == object]
FEATURES_NUM = [c for c in _todas if c not in FEATURES_CAT]
for _c in FEATURES_NUM:
    df_split[_c] = pd.to_numeric(df_split[_c], errors="coerce")
df_split[TARGET] = pd.to_numeric(df_split[TARGET], errors="coerce").astype(int)

ALL_FEATURES = FEATURES_NUM + FEATURES_CAT

_nan = [c for c in FEATURES_NUM if df_split[c].isna().any()]
if _nan:
    raise ValueError(
        f"Features numéricas com NaN na tabela de split: {_nan}. "
        f"Reexecute o notebook 03."
    )

# ── Reconstrução das partições ────────────────────────────────────────────────
_tr = df_split[df_split["_SPLIT"] == "train"]
_te = df_split[df_split["_SPLIT"] == "test"]

X_train, y_train = _tr[ALL_FEATURES].copy(), _tr[TARGET].copy()
X_test, y_test = _te[ALL_FEATURES].copy(), _te[TARGET].copy()

# A chave fica fora do X, disponível para auditar qual cliente foi classificado
# como o quê. Só ID_PESSOA: a reidentificação, se necessária, é consulta ao ERP.
ID_TEST = _te[[c for c in _IDENTIDADE if c in _te.columns]].copy()

n_pos_teste = int((y_test == 1).sum())
n_neg_teste = int((y_test == 0).sum())
N_MINORIA_TESTE = min(n_pos_teste, n_neg_teste)

print("=" * 78)
print(f"{'SPLIT CARREGADO':^78}")
print("=" * 78)
print(f"  SPLIT_HASH   : {SPLIT_HASH}")
print(f"  DATA_VERSION : {DATA_VERSION}")
print("-" * 78)
print(f"  X_train : {X_train.shape[0]:>5,} × {X_train.shape[1]:>2}   prevalência {y_train.mean():.1%}")
print(f"  X_test  : {X_test.shape[0]:>5,} × {X_test.shape[1]:>2}   prevalência {y_test.mean():.1%}")
print(f"  Features: {len(ALL_FEATURES)} ({len(FEATURES_NUM)} num + {len(FEATURES_CAT)} cat)")
print(f"    ├─ num: {', '.join(FEATURES_NUM)}")
print(f"    └─ cat: {', '.join(FEATURES_CAT)}")
print("-" * 78)
print(f"  ⚠️  Classe minoritária no teste: {N_MINORIA_TESTE} casos")
if N_MINORIA_TESTE < 20:
    print(f"      Um cliente reclassificado move a métrica ~{100/max(N_MINORIA_TESTE,1):.1f} pontos.")
    print(f"      Leia sempre o IC junto do valor, e prefira SELECAO['origem']='cv'.")
print("=" * 78)
print()
print("  ⚠️  Confira que SPLIT_HASH é o MESMO nos notebooks 04, 05 e 06 antes de")
print("      comparar resultados entre algoritmos.")

## 📐 3. Motor de Métricas

### Por que estas métricas, e não accuracy

A classe positiva é **maioria**. Nesse regime, prever "todos são risco" já entrega
accuracy alta e recall perfeito — sem que o modelo tenha aprendido nada. A classe
rara aqui é a **0** (bom pagador), e é ela que o negócio precisa identificar para não
recusar crédito a quem paga.

Por isso o conjunto inclui:

| Métrica | Por que está aqui |
| :--- | :--- |
| **MCC** | só sobe quando **ambas** as classes são bem classificadas; o baseline majoritário tem MCC = 0 por construção |
| **Balanced accuracy** | média dos recalls das duas classes — insensível à prevalência |
| **Specificity** | recall da classe rara: quantos bons pagadores foram reconhecidos |
| **AP da classe 0** | precisão média para a classe rara, que a AP tradicional ignora |
| **Brier** | qualidade da *probabilidade*, não só do rótulo — importa para as faixas de risco |

### Intervalos de confiança

Com poucos casos da classe rara no teste, **o intervalo é a parte mais informativa do
resultado**. Usamos Wilson para proporções (apropriado a `n` pequeno, ao contrário do
IC normal, que pode extrapolar `[0,1]`) e Hanley & McNeil para a AUC.

Um IC largo não é defeito do relatório: é o que os dados sustentam. Reportar três
casas decimais de um AUC pontual seria mais bonito e menos honesto.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MOTOR DE MÉTRICAS
#  A classe positiva é MAIORIA: accuracy e AUC isoladas enganam. A classe RARA é
#  a 0 (bom pagador), e é ela que o negócio precisa identificar.
# ══════════════════════════════════════════════════════════════════════════════
_Z_IC = INTERVALO_CONFIANCA["z"]


def ic_wilson(acertos: int, n: int, z: float = _Z_IC) -> tuple:
    """
    IC de proporção pelo método de Wilson.

    Preferido ao IC normal porque com n pequeno (dezenas) o normal produz limites
    fora de [0,1] e cobertura ruim justamente onde mais se precisa dele.
    """
    if n == 0:
        return (float("nan"), float("nan"))
    p = acertos / n
    den = 1 + z**2 / n
    centro = (p + z**2 / (2 * n)) / den
    margem = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return (max(0.0, centro - margem), min(1.0, centro + margem))


def ic_auc_hanley(auc: float, n_pos: int, n_neg: int, z: float = _Z_IC) -> tuple:
    """
    IC da AUC por Hanley & McNeil (1982). Retorna (low, high, se).

    Crítico neste projeto: com poucos negativos o IC é largo e mostra que a
    ordenação entre modelos pode não ter suporte estatístico nenhum.
    """
    if n_pos == 0 or n_neg == 0:
        return (float("nan"), float("nan"), float("nan"))
    q1 = auc / (2 - auc)
    q2 = 2 * auc**2 / (1 + auc)
    se = float(np.sqrt(
        (auc * (1 - auc) + (n_pos - 1) * (q1 - auc**2) + (n_neg - 1) * (q2 - auc**2))
        / (n_pos * n_neg)
    ))
    return (max(0.0, auc - z * se), min(1.0, auc + z * se), se)


def avaliar_no_teste(pipeline, X_te, y_te, threshold: float) -> dict:
    """
    Conjunto completo de métricas de holdout, para AMBAS as classes.

    threshold é aplicado sobre predict_proba. Quando != 0.5, as métricas
    dependentes de rótulo (F1, recall, MCC, matriz) mudam sem retreino — por isso
    o limiar usado é logado junto das métricas.
    """
    y_proba = pipeline.predict_proba(X_te)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    report = cast(Dict[str, Any], classification_report(
        y_te, y_pred, output_dict=True, zero_division=0))
    c1 = report.get("1", {"precision": 0, "recall": 0, "f1-score": 0})
    c0 = report.get("0", {"precision": 0, "recall": 0, "f1-score": 0})

    auc = float(roc_auc_score(y_te, y_proba))
    n_pos, n_neg = int((y_te == 1).sum()), int((y_te == 0).sum())
    auc_lo, auc_hi, auc_se = ic_auc_hanley(auc, n_pos, n_neg)

    tp = int(((y_te == 1) & (y_pred == 1)).sum())
    tn = int(((y_te == 0) & (y_pred == 0)).sum())
    rec1_lo, rec1_hi = ic_wilson(tp, n_pos)
    spec_lo, spec_hi = ic_wilson(tn, n_neg)

    # AP da classe 0 (a rara): inverte rótulo e score. A AP tradicional só olha a
    # classe positiva, que aqui é a fácil.
    ap_c0 = float(average_precision_score(1 - np.asarray(y_te), 1 - y_proba))

    return {
        "AUC": auc, "AUC_ic_low": float(auc_lo), "AUC_ic_high": float(auc_hi),
        "AUC_se": float(auc_se),
        "AP": float(average_precision_score(y_te, y_proba)), "AP_classe0": ap_c0,
        "F1": float(c1["f1-score"]), "Recall": float(c1["recall"]),
        "Recall_ic_low": float(rec1_lo), "Recall_ic_high": float(rec1_hi),
        "Precisao": float(c1["precision"]),
        "Specificity": float(c0["recall"]),
        "Specificity_ic_low": float(spec_lo), "Specificity_ic_high": float(spec_hi),
        "Precisao_classe0": float(c0["precision"]), "F1_classe0": float(c0["f1-score"]),
        "Accuracy": float(report["accuracy"]),
        "Balanced_Accuracy": float(balanced_accuracy_score(y_te, y_pred)),
        "MCC": float(matthews_corrcoef(y_te, y_pred)),
        "Brier": float(brier_score_loss(y_te, y_proba)),
        "F2": float(fbeta_score(y_te, y_pred, beta=2, zero_division=0)),
        "threshold": float(threshold),
        "y_pred": y_pred, "y_proba": y_proba,
    }


# ── Mapa chave interna → chave logada no MLflow ───────────────────────────────
# Fonte única. Não deduza a chave com .lower(): 'Recall' vira 'recall_classe1', e
# acento em cláusula order_by do MLflow é rejeitado.
MAPA_METRICA_MLFLOW: Dict[str, str] = {
    "AUC": "auc", "AUC_ic_low": "auc_ic_low", "AUC_ic_high": "auc_ic_high",
    "AUC_se": "auc_se", "AP": "ap_classe1", "AP_classe0": "ap_classe0",
    "F1": "f1_classe1", "Recall": "recall_classe1",
    "Recall_ic_low": "recall_classe1_ic_low", "Recall_ic_high": "recall_classe1_ic_high",
    "Precisao": "precision_classe1", "Specificity": "specificity_classe0",
    "Specificity_ic_low": "specificity_ic_low", "Specificity_ic_high": "specificity_ic_high",
    "Precisao_classe0": "precision_classe0", "F1_classe0": "f1_classe0",
    "Accuracy": "accuracy", "Balanced_Accuracy": "balanced_accuracy",
    "MCC": "mcc", "Brier": "brier", "F2": "f2_classe1",
}

# Métricas realmente disponíveis em cada origem de seleção. A lista de CV é menor:
# as métricas de CV não carregam Specificity, Brier nem F2.
METRICAS_DISPONIVEIS = {
    "teste": set(MAPA_METRICA_MLFLOW),
    "cv": {"AUC", "AUC_std", "AP", "F1", "Recall", "Precisao", "Balanced_Accuracy", "MCC"},
}

print("✅ Motor de métricas carregado")
print(f"   ├─ {len(MAPA_METRICA_MLFLOW)} métricas de holdout (ambas as classes)")
print(f"   ├─ IC de Wilson (proporções) e Hanley & McNeil (AUC), z={_Z_IC}")
print(f"   └─ Seleção por '{SELECAO['metrica']}' ({SELECAO['origem']})")

## 🏭 4. Fábrica de Pipeline

Nenhum hiperparâmetro é escrito nesta célula. Ela **lê o painel** e materializa os
objetos sklearn correspondentes.

A consequência prática é que o que o MLflow registra como configuração é, por
construção, o que de fato foi treinado — não existe caminho pelo qual os dois
divirjam. Um hiperparâmetro escrito no meio do notebook é um hiperparâmetro que o log
não enxerga.

> `clone()` no pré-processador: cada pipeline recebe a própria cópia não-treinada. Sem
> isso, os pipelines de um mesmo lote compartilhariam a instância por referência, e o
> `fit()` de uma variação sobrescreveria in place o `StandardScaler.mean_` já ajustado
> da outra.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  FÁBRICA DE PIPELINE
#  Não escreve hiperparâmetro nenhum: LÊ o painel e materializa os objetos.
#  É o que garante que o log do MLflow e o objeto treinado nunca divirjam.
# ══════════════════════════════════════════════════════════════════════════════
_SCALERS = {
    "standard": StandardScaler,
    "minmax": MinMaxScaler,
    "robust": RobustScaler,
    "none": None,
}


def build_preprocessor(cfg_preproc: dict, features_num: list, features_cat: list) -> ColumnTransformer:
    """Materializa o ColumnTransformer descrito no painel."""
    nome = cfg_preproc["scaler_numerico"]
    if nome not in _SCALERS:
        raise ValueError(f"scaler_numerico='{nome}' desconhecido. Opções: {list(_SCALERS)}")
    classe = _SCALERS[nome]
    step_num = "passthrough" if classe is None else classe()

    if cfg_preproc["encoder_categorico"] != "onehot":
        raise ValueError(f"encoder_categorico='{cfg_preproc['encoder_categorico']}' não implementado.")

    step_cat = OneHotEncoder(
        handle_unknown=cfg_preproc["onehot_handle_unknown"],  # categoria nova em
        drop=cfg_preproc["onehot_drop"],                      # produção não quebra
        sparse_output=False,
    )

    return ColumnTransformer(
        transformers=[
            ("num", step_num, features_num),
            ("cat", step_cat, features_cat),
        ],
        remainder=cfg_preproc["remainder"],
        verbose_feature_names_out=False,
    )


def build_pipeline(clf, cfg_preproc: dict,
                   features_num=None, features_cat=None) -> Pipeline:
    """
    Pipeline = pré-processador + classificador.

    clone() no preprocessor: cada pipeline recebe a PRÓPRIA cópia não-treinada.
    Sem isso as variações do lote compartilhariam a instância por referência, e o
    fit() de uma sobrescreveria in place o scaler já ajustado da outra —
    corrompendo silenciosamente qualquer estudo que varie as colunas de entrada.
    """
    fn = FEATURES_NUM if features_num is None else features_num
    fc = FEATURES_CAT if features_cat is None else features_cat
    prep = build_preprocessor(cfg_preproc, fn, fc)
    return Pipeline(steps=[("prep", clone(prep)), ("clf", clf)])


def build_classifier(params: dict, random_state: int = RANDOM_STATE):
    """Instancia o classificador deste notebook com os params dados."""
    p = dict(params)
    p["random_state"] = random_state
    return CLASSE_MODELO(**p)


def calcular_hash(*blocos, tamanho: int = 12) -> str:
    """SHA-256 truncado de um JSON canônico dos blocos (chaves ordenadas)."""
    return hashlib.sha256(
        json.dumps(blocos, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    ).hexdigest()[:tamanho]


def achatar(d: dict, prefixo: str = "", sep: str = ".") -> Dict[str, Any]:
    """
    Achata dicionário aninhado em pares de um nível só.

    mlflow.log_params() não aceita valores aninhados; listas viram string
    separada por vírgula para continuarem filtráveis no painel.
    """
    plano: Dict[str, Any] = {}
    for chave, valor in d.items():
        nome = f"{prefixo}{sep}{chave}" if prefixo else str(chave)
        if isinstance(valor, dict):
            plano.update(achatar(valor, nome, sep))
        elif isinstance(valor, (list, tuple)):
            plano[nome] = ", ".join(map(str, valor))
        else:
            plano[nome] = valor
    return plano


# Demonstração: pipeline com os hiperparâmetros-base do painel
_demo = build_pipeline(build_classifier(HP_BASE), PREPROC)
print("✅ Fábrica de pipeline pronta")
print(f"   ├─ Pré-proc : {PREPROC['scaler_numerico']} ({len(FEATURES_NUM)} num) + "
      f"onehot({len(FEATURES_CAT)} cat, drop={PREPROC['onehot_drop']})")
print(f"   └─ Modelo   : {CLASSE_MODELO.__name__}")
display(_demo)

## ⚙️ 5. Motor de Experimentos

Esta é a peça que substitui o loop monolítico da versão anterior. Ela executa **um
lote** — o estudo declarado no painel — e registra cada variação como um run filho do
MLflow.

### Estrutura no MLflow

```
Chopp_Cia_Experimentos                       (experimento)
└── LOTE__<estudo>__<modelo>__<split_hash>   run pai — a rodada inteira
    ├── baseline                             âncora obrigatória
    ├── <eixo>=<valor 1>                     run filho — uma variação
    ├── <eixo>=<valor 2>
    └── ...
```

### As tags que tornam o painel navegável

Cada run filho carrega:

| Tag | Serve para |
| :--- | :--- |
| `estudo` | agrupar o lote — `elegibilidade`, `regularizacao`, `ablacao`… |
| `eixo_varrido` | qual parâmetro muda dentro do lote |
| `valor_eixo` | o valor daquela variação |
| `split_hash` | **comparar entre modelos**: mesmo hash ⇒ mesmas linhas |
| `data_version` | qual carga do banco |
| `modelo` | qual algoritmo |

Com isso, no painel do Databricks:

- **"quantos clientes usar?"** → filtre `estudo='elegibilidade'`, ordene por MCC
- **"qual C usar?"** → filtre `estudo='regularizacao'`
- **"qual algoritmo vence?"** → filtre `split_hash='<hash>'` e compare `modelo`
- **"quanto do desempenho vem do vazamento?"** → filtre `estudo='ablacao'`

### O que cada run filho registra

1. **Validação cruzada** com todas as métricas do painel, fold a fold
2. **Gap treino − validação** (`cv_*_gap_overfit`) — o diagnóstico direto de
   overfitting, e o número a observar ao tunar capacidade
3. **Holdout** com intervalos de confiança para ambas as classes
4. **Varredura de limiar out-of-fold** — o corte operacional escolhido por curva, sem
   tocar no teste
5. **Modelo com assinatura** + a configuração completa como artefato JSON

> A varredura de limiar roda sobre predições **out-of-fold do treino**, não sobre o
> holdout. Escolher o corte olhando o teste é tuning feito no conjunto de teste: o
> limiar "ótimo" ali é ruído, e depois de escolhido o holdout já não mede
> generalização.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MOTOR DE EXPERIMENTOS
#
#  Executa UM LOTE (o estudo declarado no painel) como:
#     1 run PAI  →  N runs FILHOS, um por variação
#
#  Contrato de rastreabilidade de cada run:
#     · tags   → estudo, eixo_varrido, valor_eixo, split_hash, data_version, modelo
#     · params → hp.* (painel) e clf__* (o que o sklearn realmente usou)
#     · config completa como artefato JSON — reprodutível por cópia direta
# ══════════════════════════════════════════════════════════════════════════════

# Run órfão da sessão anterior capturaria tudo que viesse depois: log_metric fora
# de um run não falha — abre um run implícito que nunca é encerrado.
if mlflow.active_run() is not None:
    print(f"Encerrando run órfão: {mlflow.active_run().info.run_id}")
    mlflow.end_run()

mlflow.set_experiment(EXPERIMENT_NAME)

# Autolog DESLIGADO, por dois motivos concretos:
#   1) o MLflow desabilita o autolog durante cross_validate() — as métricas de CV
#      nunca chegariam ao servidor por essa via;
#   2) log_post_training_metrics intercepta roc_auc_score chamado após o fit e
#      loga com nomes automáticos, duplicando chaves e tornando o Compare ilegível.
mlflow.sklearn.autolog(disable=True)

TAGS_PROJETO = {
    "projeto": "Chopp & Cia",
    "instituicao": "FATEC Votorantim",
    "disciplina": "Projeto Integrador VI",
    "tipo_modelo": "classificacao_binaria",
    "framework": "scikit-learn",
    "unidade_analise": "ID_PESSOA",
}

LIMITACOES_CONHECIDAS = {
    "sem_corte_temporal": (
        "Features e alvo sao calculados sobre a MESMA janela temporal. As metricas "
        "medem capacidade de REPRODUZIR a regra de negocio, nao de prever o futuro."
    ),
    "features_derivadas_do_alvo": (
        "MEDIA_DIAS_ATRASO_PAG/COM compartilham origem aritmetica com TAXA_ATRASO_*, "
        "que constroi ALTO_RISCO. Quantificado pelo estudo 'ablacao'."
    ),
    "classe_positiva_majoritaria": (
        "A classe positiva e maioria. Accuracy e AUC isoladas enganam; compare "
        "sempre contra o baseline de classe majoritaria."
    ),
    "n_pequeno_no_teste": (
        "Poucos casos da classe minoritaria no holdout: intervalos largos. "
        "Para comparar muitas configuracoes, use SELECAO['origem']='cv'."
    ),
}


def _cv():
    """StratifiedKFold configurado pelo painel."""
    return StratifiedKFold(
        n_splits=CV["n_splits"],
        shuffle=CV["shuffle"],
        random_state=RANDOM_STATE if CV["shuffle"] else None,
    )


def _logar_metricas_teste(metricas: dict, prefixo: str = "test") -> None:
    """Loga só os escalares — y_pred/y_proba são arrays, não métricas."""
    for chave_pt, chave_ascii in MAPA_METRICA_MLFLOW.items():
        valor = metricas.get(chave_pt)
        if isinstance(valor, float) and not np.isnan(valor):
            mlflow.log_metric(f"{prefixo}_{chave_ascii}", valor)


def _sweep_threshold(pipeline, X_tr, y_tr) -> tuple:
    """
    Varre a grade de limiares sobre predições OUT-OF-FOLD do treino.

    Por que out-of-fold e não no holdout: escolher o corte olhando o test_* é
    tuning feito no conjunto de teste. Com poucos negativos, o limiar "ótimo" ali
    é ruído — e depois de escolhido, o holdout já não mede generalização.
    Aqui cada previsão vem de um fold que não viu aquele cliente no treino.

    Retorna (threshold_otimo, mcc_no_otimo).
    """
    grade = np.round(
        np.arange(THRESHOLD["sweep_inicio"],
                  THRESHOLD["sweep_fim"] + THRESHOLD["sweep_passo"] / 2,
                  THRESHOLD["sweep_passo"]),
        4,
    )
    proba_oof = cross_val_predict(
        pipeline, X_tr, y_tr, cv=_cv(), method="predict_proba", n_jobs=CV["n_jobs"],
    )[:, 1]
    y_oof = np.asarray(y_tr)

    melhor = (float(THRESHOLD["classificacao"]), -np.inf)
    for passo, thr in enumerate(grade):
        pred = (proba_oof >= thr).astype(int)
        rep = cast(Dict[str, Any], classification_report(
            y_oof, pred, output_dict=True, zero_division=0))
        c1 = rep.get("1", {"precision": 0, "recall": 0})
        c0 = rep.get("0", {"recall": 0})
        mcc = float(matthews_corrcoef(y_oof, pred))

        mlflow.log_metric("sweep_threshold", float(thr), step=passo)
        mlflow.log_metric("sweep_oof_mcc", mcc, step=passo)
        mlflow.log_metric("sweep_oof_recall_classe1", float(c1["recall"]), step=passo)
        mlflow.log_metric("sweep_oof_precision_classe1", float(c1["precision"]), step=passo)
        mlflow.log_metric("sweep_oof_specificity_classe0", float(c0["recall"]), step=passo)
        mlflow.log_metric("sweep_oof_balanced_accuracy",
                          float(balanced_accuracy_score(y_oof, pred)), step=passo)

        if mcc > melhor[1]:
            melhor = (float(thr), mcc)

    mlflow.log_metric("threshold_otimo_oof", melhor[0])
    mlflow.log_metric("mcc_no_threshold_otimo_oof", melhor[1])
    return melhor


def treinar_variacao(nome_variacao: str, hp: dict, eixo: str, valor,
                     features_num=None, features_cat=None,
                     X_tr=None, y_tr=None, X_te=None, y_te=None) -> dict:
    """
    Treina UMA variação e registra tudo num run filho do MLflow.

    Parâmetros de dados são opcionais: o estudo de elegibilidade passa
    subconjuntos das partições, e o de ablação passa listas de features
    reduzidas. Os demais estudos usam o split completo.

    Retorna o dicionário de resultado, também acumulado em RESULTADOS.
    """
    X_tr = X_train if X_tr is None else X_tr
    y_tr = y_train if y_tr is None else y_tr
    X_te = X_test if X_te is None else X_te
    y_te = y_test if y_te is None else y_te
    fn = FEATURES_NUM if features_num is None else features_num
    fc = FEATURES_CAT if features_cat is None else features_cat

    clf = build_classifier(hp)
    pipeline = build_pipeline(clf, PREPROC, fn, fc)
    cols = fn + fc

    # Hash da configuração desta variação: identifica-a de forma estável entre
    # execuções, sem depender do nome legível.
    config_hash = calcular_hash(hp, PREPROC, CV, SPLIT_HASH, cols)

    with mlflow.start_run(run_name=nome_variacao, nested=True) as run:
        # ── TAGS: os eixos pelos quais se filtra e agrupa no painel ───────────
        mlflow.set_tags({
            **TAGS_PROJETO,
            "modelo": NOME_MODELO,
            "algoritmo": CLASSE_MODELO.__name__,
            "estudo": ESTUDO,
            "eixo_varrido": eixo,
            "valor_eixo": str(valor),
            "split_hash": SPLIT_HASH,
            "data_version": DATA_VERSION,
            "config_hash": config_hash,
            "tipo_run": "variacao",
            "etapa": "modelagem",
        })

        # ── PARAMS ────────────────────────────────────────────────────────────
        # (a) o painel — a INTENÇÃO declarada
        mlflow.log_params(achatar(hp, "hp"))
        mlflow.log_params(achatar(PREPROC, "preproc"))
        mlflow.log_params(achatar(CV, "cv"))
        mlflow.log_params({
            "estudo": ESTUDO, "eixo_varrido": eixo, "valor_eixo": str(valor),
            "split_hash": SPLIT_HASH, "data_version": DATA_VERSION,
            "config_hash": config_hash, "random_state": RANDOM_STATE,
            "threshold_classificacao": THRESHOLD["classificacao"],
            "n_features": len(cols), "features": ", ".join(cols),
            "n_train": len(X_tr), "n_test": len(X_te),
            "n_test_negativos": int((y_te == 0).sum()),
            "prevalencia_treino": round(float(y_tr.mean()), 4),
            "prevalencia_teste": round(float(y_te.mean()), 4),
        })
        # (b) params efetivos do estimador — o que o sklearn REALMENTE usou,
        #     defaults incluídos. Comparar (a) com (b) revela qualquer default
        #     que tenha entrado sem passar pelo painel.
        for k, v in clf.get_params().items():
            mlflow.log_param(f"clf__{k}", v)

        # ── VALIDAÇÃO CRUZADA ─────────────────────────────────────────────────
        cv_result = cross_validate(
            pipeline, X_tr[cols], y_tr, cv=_cv(), scoring=SCORING_CV,
            return_train_score=CV["return_train_score"], n_jobs=CV["n_jobs"],
        )

        metricas_cv = {}
        for metrica in SCORING_CV:
            scores_val = cv_result[f"test_{metrica}"]
            metricas_cv[metrica] = float(scores_val.mean())
            mlflow.log_metric(f"cv_{metrica}_mean", float(scores_val.mean()))
            mlflow.log_metric(f"cv_{metrica}_std", float(scores_val.std()))
            for i, s in enumerate(scores_val):
                mlflow.log_metric(f"fold_{metrica}", float(s), step=i)
            # Gap treino−validação: o diagnóstico direto de overfitting, e o
            # número a observar ao tunar capacidade (max_depth, C, min_samples).
            if CV["return_train_score"]:
                gap = float(cv_result[f"train_{metrica}"].mean() - scores_val.mean())
                mlflow.log_metric(f"cv_{metrica}_gap_overfit", gap)

        metricas_cv_pt = {
            "AUC": metricas_cv["roc_auc"],
            "AUC_std": float(cv_result["test_roc_auc"].std()),
            "AP": metricas_cv["average_precision"],
            "F1": metricas_cv["f1"], "Recall": metricas_cv["recall"],
            "Precisao": metricas_cv["precision"],
            "Balanced_Accuracy": metricas_cv["balanced_accuracy"],
            "MCC": metricas_cv["matthews_corrcoef"],
        }

        # ── TREINO FINAL + HOLDOUT ────────────────────────────────────────────
        pipeline.fit(X_tr[cols], y_tr)
        metricas_teste = avaliar_no_teste(pipeline, X_te[cols], y_te,
                                          THRESHOLD["classificacao"])
        _logar_metricas_teste(metricas_teste)

        # ── VARREDURA DE LIMIAR (out-of-fold) ─────────────────────────────────
        thr_otimo = float(THRESHOLD["classificacao"])
        try:
            thr_otimo, _ = _sweep_threshold(pipeline, X_tr[cols], y_tr)
        except Exception as e:
            print(f"      ⚠️  sweep indisponível ({type(e).__name__})")

        # ── MATRIZ DE CONFUSÃO em contagens ABSOLUTAS ─────────────────────────
        # Absoluto, não normalizado: o percentual esconde que a linha dos
        # negativos tem pouquíssimos casos.
        tn, fp, fn_, tp = confusion_matrix(y_te, metricas_teste["y_pred"]).ravel()
        mlflow.log_metrics({
            "cm_verdadeiro_negativo": int(tn), "cm_falso_positivo": int(fp),
            "cm_falso_negativo": int(fn_),   # custo alto: risco não detectado
            "cm_verdadeiro_positivo": int(tp),
        })

        # ── MODELO + assinatura (contrato de entrada/saída) ───────────────────
        try:
            assinatura = infer_signature(X_tr[cols], pipeline.predict(X_tr[cols]))
            mlflow.sklearn.log_model(
                sk_model=pipeline, name="modelo",
                signature=assinatura, input_example=X_tr[cols].head(3),
            )
        except Exception as e:
            print(f"      ⚠️  log_model indisponível ({type(e).__name__})")

        # ── ARTEFATOS ─────────────────────────────────────────────────────────
        mlflow.log_dict(
            cast(Dict[str, Any], classification_report(
                y_te, metricas_teste["y_pred"], output_dict=True, zero_division=0)),
            "relatorio_classificacao.json")
        mlflow.log_dict(LIMITACOES_CONHECIDAS, "limitacoes_metodologicas.json")
        # Basta copiar este JSON de volta ao painel para reproduzir a variação.
        mlflow.log_dict({
            "estudo": ESTUDO, "variacao": nome_variacao,
            "eixo_varrido": eixo, "valor_eixo": str(valor),
            "hiperparametros": hp, "preproc": PREPROC, "cv": CV,
            "threshold": THRESHOLD, "selecao": SELECAO,
            "split_hash": SPLIT_HASH, "data_version": DATA_VERSION,
            "features": cols, "config_hash": config_hash,
            "params_efetivos_sklearn": {k: str(v) for k, v in clf.get_params().items()},
        }, "config_variacao.json")

        run_id = run.info.run_id

    return {
        "variacao": nome_variacao, "eixo": eixo, "valor": valor,
        "hp": hp, "features": cols, "pipeline": pipeline,
        "metricas_cv": metricas_cv_pt, "metricas_teste": metricas_teste,
        "threshold_otimo_oof": thr_otimo,
        "config_hash": config_hash, "run_id": run_id,
    }


def treinar_baseline(X_tr=None, y_tr=None, X_te=None, y_te=None) -> dict:
    """
    Baseline de classe majoritária: a âncora obrigatória da comparação.

    Sem ela nenhuma métrica significa nada — prever "sempre risco" já entrega
    accuracy alta e recall perfeito. O baseline tem MCC = 0 por construção, o que
    dá um piso claro para ler o MCC dos candidatos.
    """
    X_tr = X_train if X_tr is None else X_tr
    y_tr = y_train if y_tr is None else y_tr
    X_te = X_test if X_te is None else X_te
    y_te = y_test if y_te is None else y_te

    with mlflow.start_run(run_name="baseline (classe majoritária)", nested=True):
        dummy = build_pipeline(
            DummyClassifier(strategy=BASELINE["strategy"], random_state=RANDOM_STATE),
            PREPROC,
        )
        dummy.fit(X_tr[ALL_FEATURES], y_tr)
        met = avaliar_no_teste(dummy, X_te[ALL_FEATURES], y_te, THRESHOLD["classificacao"])

        mlflow.set_tags({
            **TAGS_PROJETO, "modelo": "Baseline", "algoritmo": "DummyClassifier",
            "estudo": ESTUDO, "eixo_varrido": "baseline", "valor_eixo": BASELINE["strategy"],
            "split_hash": SPLIT_HASH, "data_version": DATA_VERSION,
            "tipo_run": "baseline", "etapa": "modelagem",
        })
        # Mesmos params de forma dos candidatos: sem eles a linha do baseline fica
        # com células vazias no 'Compare' do Databricks.
        mlflow.log_params({
            "estudo": ESTUDO, "split_hash": SPLIT_HASH, "data_version": DATA_VERSION,
            "n_train": len(X_tr), "n_test": len(X_te),
            "n_test_negativos": int((y_te == 0).sum()),
            "prevalencia_treino": round(float(y_tr.mean()), 4),
            "prevalencia_teste": round(float(y_te.mean()), 4),
            "strategy": BASELINE["strategy"],
        })
        _logar_metricas_teste(met)

    return met


print("✅ Motor de experimentos carregado")
print(f"   ├─ treinar_variacao()  → 1 run filho por variação")
print(f"   ├─ treinar_baseline()  → âncora da comparação")
print(f"   └─ Experimento MLflow  : {EXPERIMENT_NAME}")

## 🧪 6. Montagem do Lote

O painel declarou **qual estudo** rodar. Esta célula traduz essa declaração numa lista
concreta de variações — e é aqui que os estudos que mexem nos *dados* (elegibilidade,
janela temporal, ablação) se distinguem dos que mexem só nos *hiperparâmetros*.

| Estudo | O que varia | O que responde |
| :--- | :--- | :--- |
| `hiperparametros` | um parâmetro do algoritmo | qual configuração aprende melhor |
| `elegibilidade` | `min_compras` (subconjunto das linhas) | quantos clientes vale a pena usar |
| `janela_temporal` | recência mínima (subconjunto das linhas) | qual histórico generaliza melhor |
| `ablacao` | remove as features derivadas do alvo | quanto do desempenho vem do vazamento |
| `baseline_unico` | nada — uma configuração só | rodada de referência |

> Os estudos `elegibilidade` e `janela_temporal` **filtram as linhas do split já
> congelado**, em vez de refazer a partição. Assim, um cliente que está no teste
> continua no teste em todas as variações do lote — e a diferença de métrica vem do
> filtro, não de um sorteio novo.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MONTAGEM DO LOTE
#  Traduz ESTUDO + GRADE_ESTUDO numa lista concreta de variações a executar.
# ══════════════════════════════════════════════════════════════════════════════

def _filtrar_por_min_compras(minimo: int):
    """
    Subconjunto das partições por histórico mínimo.

    Filtra as linhas do split JÁ CONGELADO, em vez de reparticionar: um cliente
    que está no teste continua no teste em todas as variações, então a diferença
    de métrica vem do filtro — não de um sorteio novo.
    """
    if "FREQUENCIA_COMPRAS" not in X_train.columns:
        raise KeyError(
            "FREQUENCIA_COMPRAS não está entre as features — o estudo de "
            "elegibilidade precisa dela para filtrar. Inclua-a no notebook 03."
        )
    m_tr = X_train["FREQUENCIA_COMPRAS"] > minimo
    m_te = X_test["FREQUENCIA_COMPRAS"] > minimo
    return X_train[m_tr], y_train[m_tr], X_test[m_te], y_test[m_te]


def _filtrar_por_recencia(dias_max: int):
    """
    Subconjunto por recência: mantém quem comprou nos últimos `dias_max` dias.

    Mesma lógica do filtro anterior — recorta o split congelado, preservando a
    atribuição treino/teste de cada cliente.
    """
    if "DIAS_DESDE_ULTIMA_COMPRA" not in X_train.columns:
        raise KeyError(
            "DIAS_DESDE_ULTIMA_COMPRA não está entre as features — o estudo de "
            "janela temporal precisa dela."
        )
    m_tr = X_train["DIAS_DESDE_ULTIMA_COMPRA"] <= dias_max
    m_te = X_test["DIAS_DESDE_ULTIMA_COMPRA"] <= dias_max
    return X_train[m_tr], y_train[m_tr], X_test[m_te], y_test[m_te]


def montar_lote() -> List[dict]:
    """
    Constrói a lista de variações do lote declarado no painel.

    Cada item traz o que treinar_variacao() precisa: nome, hiperparâmetros, eixo,
    valor e — quando o estudo mexe nos dados — as partições e features a usar.
    """
    variacoes: List[dict] = []

    # ── Estudo de hiperparâmetros: varre um parâmetro do algoritmo ────────────
    if ESTUDO == "hiperparametros":
        eixo = GRADE_ESTUDO["parametro"]
        for valor in GRADE_ESTUDO["valores"]:
            hp = {**HP_BASE, eixo: valor}
            variacoes.append({
                "nome": f"{eixo}={valor}", "hp": hp, "eixo": eixo, "valor": valor,
            })

    # ── Grade cruzada: produto cartesiano de dois ou mais parâmetros ──────────
    # Use com parcimônia: com poucos casos da classe rara, varrer muitas
    # combinações otimiza ruído — o resultado parece melhor sem ser melhor.
    elif ESTUDO == "grade_cruzada":
        nomes = list(GRADE_ESTUDO["parametros"].keys())
        for combo in itertools.product(*GRADE_ESTUDO["parametros"].values()):
            hp = {**HP_BASE, **dict(zip(nomes, combo))}
            rotulo = " · ".join(f"{k}={v}" for k, v in zip(nomes, combo))
            variacoes.append({
                "nome": rotulo, "hp": hp,
                "eixo": "+".join(nomes), "valor": rotulo,
            })

    # ── Estudo de elegibilidade: quantos clientes usar ────────────────────────
    elif ESTUDO == "elegibilidade":
        for minimo in GRADE_ESTUDO["valores"]:
            Xtr, ytr, Xte, yte = _filtrar_por_min_compras(minimo)
            # Um estrato vazio quebra a estratificação da CV com mensagem obscura.
            if min(int((ytr == 0).sum()), int((ytr == 1).sum())) < CV["n_splits"]:
                print(f"   ⏭️  min_compras>{minimo}: classe rara com menos casos "
                      f"que folds ({CV['n_splits']}) — variação pulada.")
                continue
            variacoes.append({
                "nome": f"min_compras>{minimo}", "hp": dict(HP_BASE),
                "eixo": "min_compras", "valor": minimo,
                "dados": (Xtr, ytr, Xte, yte),
            })

    # ── Estudo de janela temporal: qual histórico generaliza melhor ───────────
    elif ESTUDO == "janela_temporal":
        for dias in GRADE_ESTUDO["valores"]:
            Xtr, ytr, Xte, yte = _filtrar_por_recencia(dias)
            if min(int((ytr == 0).sum()), int((ytr == 1).sum())) < CV["n_splits"]:
                print(f"   ⏭️  recência ≤{dias}d: classe rara com menos casos "
                      f"que folds — variação pulada.")
                continue
            variacoes.append({
                "nome": f"recencia<={dias}d", "hp": dict(HP_BASE),
                "eixo": "dias_recencia_max", "valor": dias,
                "dados": (Xtr, ytr, Xte, yte),
            })

    # ── Estudo de ablação: quanto do desempenho vem do vazamento ──────────────
    # As features MEDIA_DIAS_ATRASO_* compartilham origem aritmética com o alvo.
    # Este estudo mede exatamente quanto da métrica depende delas.
    elif ESTUDO == "ablacao":
        for conjunto in GRADE_ESTUDO["conjuntos"]:
            remover = set(conjunto["remover"])
            fn = [c for c in FEATURES_NUM if c not in remover]
            fc = [c for c in FEATURES_CAT if c not in remover]
            if not fn and not fc:
                print(f"   ⏭️  '{conjunto['nome']}' removeria todas as features — pulada.")
                continue
            variacoes.append({
                "nome": conjunto["nome"], "hp": dict(HP_BASE),
                "eixo": "features_removidas",
                "valor": ", ".join(conjunto["remover"]) or "nenhuma",
                "features": (fn, fc),
            })

    # ── Rodada única de referência ────────────────────────────────────────────
    elif ESTUDO == "baseline_unico":
        variacoes.append({
            "nome": "configuracao_base", "hp": dict(HP_BASE),
            "eixo": "nenhum", "valor": "base",
        })

    else:
        raise ValueError(
            f"ESTUDO='{ESTUDO}' desconhecido. Opções: hiperparametros, "
            f"grade_cruzada, elegibilidade, janela_temporal, ablacao, baseline_unico."
        )

    if not variacoes:
        raise ValueError(
            "Nenhuma variação sobreviveu à montagem do lote — todas foram puladas "
            "por falta de casos na classe rara. Afrouxe a grade do estudo."
        )
    return variacoes


LOTE = montar_lote()

print("=" * 78)
print(f"{'LOTE MONTADO':^78}")
print("=" * 78)
print(f"  Estudo   : {ESTUDO}")
print(f"  Modelo   : {NOME_MODELO}")
print(f"  Variações: {len(LOTE)}")
print("-" * 78)
for i, v in enumerate(LOTE, 1):
    _extra = ""
    if "dados" in v:
        _extra = f"   (n_train={len(v['dados'][0])}, n_test={len(v['dados'][2])})"
    elif "features" in v:
        _extra = f"   ({len(v['features'][0]) + len(v['features'][1])} features)"
    print(f"   {i:>2}. {v['nome']:<40}{_extra}")
print("=" * 78)

## 🚀 7. Execução do Lote

Um run pai para o lote, um run filho por variação, mais o baseline. Ao final, a
seleção do campeão usa **o critério declarado no painel** — em um só lugar, lido
também pelas seções de avaliação e exportação.

> Na versão anterior, a Fase 5 elegia por MCC e a Fase 7 reelegia por AUC: as duas
> podiam apontar modelos diferentes, e o PKL promovido não era o campeão anunciado.
> Aqui há um único critério.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  EXECUÇÃO DO LOTE
# ══════════════════════════════════════════════════════════════════════════════
NOME_LOTE = (
    f"LOTE__{ESTUDO}__{SLUG_MODELO}__{SPLIT_HASH}"
    f"__{datetime.now().strftime('%Y%m%d_%H%M')}"
)

RESULTADOS: List[dict] = []
BASELINE_METRICAS = None

print(f"🚀 Executando lote: {NOME_LOTE}\n")
print(f"{'─'*100}")

with mlflow.start_run(run_name=NOME_LOTE) as run_pai:
    RUN_PAI_ID = run_pai.info.run_id

    mlflow.set_tags({
        **TAGS_PROJETO,
        "modelo": NOME_MODELO,
        "algoritmo": CLASSE_MODELO.__name__,
        "estudo": ESTUDO,
        "split_hash": SPLIT_HASH,
        "data_version": DATA_VERSION,
        "tipo_run": "lote",
        "etapa": "modelagem",
    })
    mlflow.log_params({
        "estudo": ESTUDO,
        "nota_estudo": NOTA_ESTUDO,
        "split_hash": SPLIT_HASH,
        "data_version": DATA_VERSION,
        "tabela_split": TABELA_SPLIT or CSV_SPLIT,
        "n_variacoes": len(LOTE),
        "modelo": NOME_MODELO,
        "random_state": RANDOM_STATE,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "n_test_negativos": n_neg_teste,
        "features": ", ".join(ALL_FEATURES),
        "criterio_selecao": f"{SELECAO['metrica']} ({SELECAO['origem']})",
        **{f"meta.{k}": v for k, v in METAS_KPI.items()},
    })
    mlflow.log_dict(LIMITACOES_CONHECIDAS, "limitacoes_metodologicas.json")
    mlflow.log_dict({
        "estudo": ESTUDO, "nota": NOTA_ESTUDO,
        "grade": GRADE_ESTUDO, "hp_base": HP_BASE,
        "preproc": PREPROC, "cv": CV, "threshold": THRESHOLD,
        "selecao": SELECAO, "split_hash": SPLIT_HASH,
        "data_version": DATA_VERSION, "features": ALL_FEATURES,
    }, "config_lote.json")

    # ── Lineage: liga o run aos dados exatos que o produziram ─────────────────
    try:
        mlflow.log_input(
            mlflow.data.from_pandas(
                pd.concat([X_train, y_train], axis=1), targets=TARGET,
                name=f"split_{SPLIT_HASH}_train"),
            context="training")
        mlflow.log_input(
            mlflow.data.from_pandas(
                pd.concat([X_test, y_test], axis=1), targets=TARGET,
                name=f"split_{SPLIT_HASH}_test"),
            context="testing")
    except Exception as e:
        print(f"   ⚠️  Lineage indisponível ({type(e).__name__})")

    # ── BASELINE: a âncora obrigatória ────────────────────────────────────────
    if BASELINE["ativo"]:
        BASELINE_METRICAS = treinar_baseline()
        print(f"  📏 {'baseline (maioria)':<42} "
              f"Acc={BASELINE_METRICAS['Accuracy']:.3f}  "
              f"BalAcc={BASELINE_METRICAS['Balanced_Accuracy']:.3f}  "
              f"MCC={BASELINE_METRICAS['MCC']:.3f}")
        print(f"{'─'*100}")

    # ── VARIAÇÕES ─────────────────────────────────────────────────────────────
    for v in LOTE:
        dados = v.get("dados", (None, None, None, None))
        feats = v.get("features", (None, None))
        try:
            res = treinar_variacao(
                v["nome"], v["hp"], v["eixo"], v["valor"],
                features_num=feats[0], features_cat=feats[1],
                X_tr=dados[0], y_tr=dados[1], X_te=dados[2], y_te=dados[3],
            )
        except Exception as e:
            # Uma variação que falha não deve derrubar o lote inteiro: as demais
            # continuam e o painel mostra o que foi possível medir.
            print(f"  ❌ {v['nome']:<42} FALHOU: {type(e).__name__}: {e}")
            continue

        RESULTADOS.append(res)
        cv_m, te_m = res["metricas_cv"], res["metricas_teste"]
        print(f"  ✅ {v['nome']:<42} "
              f"CV MCC={cv_m['MCC']:.3f}  "
              f"Teste MCC={te_m['MCC']:.3f}  "
              f"AUC={te_m['AUC']:.3f}[{te_m['AUC_ic_low']:.2f}–{te_m['AUC_ic_high']:.2f}]  "
              f"BalAcc={te_m['Balanced_Accuracy']:.3f}")

    # ══════════════════════════════════════════════════════════════════════════
    #  SELEÇÃO DO CAMPEÃO — critério único, lido do painel
    #  Definido em UM lugar e reutilizado pela avaliação e pela exportação.
    # ══════════════════════════════════════════════════════════════════════════
    def selecionar_campeao(resultados: List[dict]) -> dict:
        """Retorna o resultado vencedor segundo o critério declarado no painel."""
        if not resultados:
            raise RuntimeError("Nenhuma variação treinou com sucesso.")
        chave = "metricas_teste" if SELECAO["origem"] == "teste" else "metricas_cv"
        sinal = 1 if SELECAO["maior_melhor"] else -1

        def _ordem(r):
            m = r[chave]
            return (sinal * m.get(SELECAO["metrica"], -np.inf),
                    sinal * m.get(SELECAO["criterio_desempate"], -np.inf))

        return max(resultados, key=_ordem)

    CAMPEAO = selecionar_campeao(RESULTADOS)
    _met_camp = CAMPEAO["metricas_teste"]

    mlflow.set_tags({
        "melhor_variacao": CAMPEAO["variacao"],
        "criterio_selecao": f"{SELECAO['metrica']} ({SELECAO['origem']})",
    })
    _m_camp = {
        "melhor_test_mcc": _met_camp["MCC"],
        "melhor_test_auc": _met_camp["AUC"],
        "melhor_test_balanced_accuracy": _met_camp["Balanced_Accuracy"],
        "melhor_test_recall_classe1": _met_camp["Recall"],
        "melhor_test_specificity_classe0": _met_camp["Specificity"],
        "melhor_cv_mcc": CAMPEAO["metricas_cv"]["MCC"],
        "n_variacoes_executadas": float(len(RESULTADOS)),
    }
    if BASELINE_METRICAS is not None:
        _m_camp.update({
            "baseline_accuracy": BASELINE_METRICAS["Accuracy"],
            "baseline_mcc": BASELINE_METRICAS["MCC"],
            "ganho_mcc_sobre_baseline": _met_camp["MCC"] - BASELINE_METRICAS["MCC"],
            "ganho_balacc_sobre_baseline": (
                _met_camp["Balanced_Accuracy"] - BASELINE_METRICAS["Balanced_Accuracy"]),
        })
    mlflow.log_metrics(_m_camp)

print(f"{'─'*100}")
print(f"\n🏆 Campeão do lote por {SELECAO['metrica']} ({SELECAO['origem']}): "
      f"{CAMPEAO['variacao']}")
print(f"   ├─ MCC      : {_met_camp['MCC']:.4f}" +
      (f"   (baseline: {BASELINE_METRICAS['MCC']:.4f})" if BASELINE_METRICAS else ""))
print(f"   ├─ BalAcc   : {_met_camp['Balanced_Accuracy']:.4f}" +
      (f"   (baseline: {BASELINE_METRICAS['Balanced_Accuracy']:.4f})" if BASELINE_METRICAS else ""))
print(f"   ├─ AUC      : {_met_camp['AUC']:.4f} "
      f"[{_met_camp['AUC_ic_low']:.3f} – {_met_camp['AUC_ic_high']:.3f}]")
print(f"   ├─ Thr ótimo: {CAMPEAO['threshold_otimo_oof']:.2f} (out-of-fold)")
print(f"   └─ Run pai  : {RUN_PAI_ID}")
print(f"\n💡 No Databricks: Experiments → expanda '{NOME_LOTE}' e use 'Compare'.")
print(f"   Para comparar ENTRE modelos, filtre por tags.split_hash = '{SPLIT_HASH}'.")

## 📊 8. Análise do Lote

A tabela e os gráficos respondem à pergunta que motivou o estudo. Três leituras
importam mais que a métrica de campeão:

1. **A curva ao longo do eixo** — onde o ganho satura, e a partir de onde só se paga
   complexidade sem retorno.
2. **O gap treino − validação** — quanto o modelo decora. É o número a observar ao
   aumentar capacidade, não a métrica de validação isolada.
3. **A distância até o baseline** — um MCC de 0,3 pode ser excelente ou irrelevante
   dependendo de onde o baseline está.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  ANÁLISE DO LOTE
# ══════════════════════════════════════════════════════════════════════════════
df_lote = pd.DataFrame([{
    "variação": r["variacao"],
    "valor_eixo": r["valor"],
    "CV MCC": r["metricas_cv"]["MCC"],
    "CV AUC": r["metricas_cv"]["AUC"],
    "CV AUC ±": r["metricas_cv"]["AUC_std"],
    "CV BalAcc": r["metricas_cv"]["Balanced_Accuracy"],
    "Teste MCC": r["metricas_teste"]["MCC"],
    "Teste AUC": r["metricas_teste"]["AUC"],
    "AUC IC low": r["metricas_teste"]["AUC_ic_low"],
    "AUC IC high": r["metricas_teste"]["AUC_ic_high"],
    "Teste BalAcc": r["metricas_teste"]["Balanced_Accuracy"],
    "Recall (risco)": r["metricas_teste"]["Recall"],
    "Specificity (bom)": r["metricas_teste"]["Specificity"],
    "Brier": r["metricas_teste"]["Brier"],
    "Thr ótimo OOF": r["threshold_otimo_oof"],
    "n features": len(r["features"]),
} for r in RESULTADOS])

_ordem_col = SELECAO["metrica"]
_col_ord = f"{'Teste' if SELECAO['origem'] == 'teste' else 'CV'} {_ordem_col}"
if _col_ord in df_lote.columns:
    df_lote = df_lote.sort_values(_col_ord, ascending=not SELECAO["maior_melhor"])

display(
    df_lote.style
    .format({c: "{:.4f}" for c in df_lote.columns if df_lote[c].dtype.kind == "f"})
    .background_gradient(subset=[c for c in ["Teste MCC", "CV MCC"] if c in df_lote.columns],
                         cmap="RdYlGn")
    .hide(axis="index")
    .set_caption(f"Lote '{ESTUDO}' · {NOME_MODELO} · split {SPLIT_HASH}")
)

# ── Comparação com o baseline ─────────────────────────────────────────────────
if BASELINE_METRICAS is not None:
    print("\n" + "=" * 78)
    print(f"{'A COMPARAÇÃO QUE IMPORTA: CAMPEÃO × BASELINE':^78}")
    print("=" * 78)
    print(f"\n  {'Métrica':<22} {'Baseline':>11} {'Campeão':>11} {'Ganho':>11}")
    print("  " + "─" * 58)
    for rot, chave in [("Accuracy", "Accuracy"), ("Balanced accuracy", "Balanced_Accuracy"),
                       ("MCC", "MCC"), ("Recall (risco)", "Recall"),
                       ("Specificity (bom)", "Specificity"), ("AUC", "AUC")]:
        b, c = BASELINE_METRICAS[chave], _met_camp[chave]
        print(f"  {rot:<22} {b:>11.4f} {c:>11.4f} {c-b:>+11.4f}")
    print("  " + "─" * 58)
    print(f"\n  Accuracy do baseline = prevalência da classe majoritária. Um modelo")
    print(f"  que não a supere não aprendeu nada — e o MCC do baseline é 0 por")
    print(f"  construção, o que dá o piso claro para ler o MCC do campeão.")

In [ ]:
# ─── Gráficos do lote ─────────────────────────────────────────────────────────
# Só faz sentido traçar curva quando o eixo é numérico e há mais de um ponto.
_eixo_numerico = (
    len(RESULTADOS) > 1
    and all(isinstance(r["valor"], (int, float, np.integer, np.floating))
            for r in RESULTADOS)
)

if _eixo_numerico:
    _ord = sorted(RESULTADOS, key=lambda r: r["valor"])
    xs = [r["valor"] for r in _ord]
    eixo_nome = _ord[0]["eixo"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Lote '{ESTUDO}' · {NOME_MODELO} · split {SPLIT_HASH}",
                 fontsize=14, fontweight="bold")

    # (a) CV vs teste: onde a curva satura
    ax = axes[0]
    ax.plot(xs, [r["metricas_cv"]["MCC"] for r in _ord],
            marker="o", lw=2.2, label="CV (validação)", color="#3498DB")
    ax.plot(xs, [r["metricas_teste"]["MCC"] for r in _ord],
            marker="s", lw=2.2, ls="--", label="Teste (holdout)", color="#E74C3C")
    if BASELINE_METRICAS is not None:
        ax.axhline(BASELINE_METRICAS["MCC"], ls=":", color="#7F8C8D", lw=1.5,
                   label=f"baseline ({BASELINE_METRICAS['MCC']:.2f})")
    ax.set_xlabel(eixo_nome)
    ax.set_ylabel("MCC")
    ax.set_title("Desempenho ao longo do eixo", fontsize=11, fontweight="bold")
    ax.legend(fontsize=8.5)
    ax.grid(alpha=0.3)

    # (b) AUC com banda de IC: mostra quando a ordenação não tem suporte
    ax = axes[1]
    aucs = [r["metricas_teste"]["AUC"] for r in _ord]
    los = [r["metricas_teste"]["AUC_ic_low"] for r in _ord]
    his = [r["metricas_teste"]["AUC_ic_high"] for r in _ord]
    ax.plot(xs, aucs, marker="o", lw=2.2, color="#9B59B6")
    ax.fill_between(xs, los, his, alpha=0.2, color="#9B59B6", label="IC 95%")
    ax.axhline(0.5, ls=":", color="#7F8C8D", lw=1.3, label="acaso (0,5)")
    ax.set_xlabel(eixo_nome)
    ax.set_ylabel("AUC")
    ax.set_title("AUC no teste com IC 95%", fontsize=11, fontweight="bold")
    ax.legend(fontsize=8.5)
    ax.grid(alpha=0.3)

    # (c) Gap treino−validação: o diagnóstico de overfitting
    ax = axes[2]
    gaps = [r["metricas_cv"]["AUC"] - r["metricas_teste"]["AUC"] for r in _ord]
    cores = ["#E74C3C" if g > 0.10 else "#F39C12" if g > 0.05 else "#2ECC71" for g in gaps]
    ax.bar(range(len(xs)), gaps, color=cores, alpha=0.9)
    ax.axhline(0, color="#2C3E50", lw=1)
    ax.set_xticks(range(len(xs)))
    ax.set_xticklabels([str(v) for v in xs], rotation=35, ha="right", fontsize=8.5)
    ax.set_xlabel(eixo_nome)
    ax.set_ylabel("AUC CV − AUC teste")
    ax.set_title("Distância entre validação e holdout", fontsize=11, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"\n  Leitura do painel:")
    print(f"   (a) onde a curva satura, aumentar {eixo_nome} só paga complexidade;")
    print(f"   (b) bandas de IC que se sobrepõem ⇒ a ordenação entre variações")
    print(f"       não tem suporte estatístico — não escolha por décimos;")
    print(f"   (c) barras altas ⇒ a validação está otimista frente ao holdout.")
else:
    # Eixo categórico (ablação, grade cruzada): barras comparativas
    fig, ax = plt.subplots(figsize=(max(9, 1.7 * len(RESULTADOS)), 5.5))
    nomes = [r["variacao"] for r in RESULTADOS]
    x = np.arange(len(nomes))
    largura = 0.38
    ax.bar(x - largura/2, [r["metricas_cv"]["MCC"] for r in RESULTADOS], largura,
           label="CV MCC", color="#3498DB", alpha=0.9)
    ax.bar(x + largura/2, [r["metricas_teste"]["MCC"] for r in RESULTADOS], largura,
           label="Teste MCC", color="#E74C3C", alpha=0.9)
    if BASELINE_METRICAS is not None:
        ax.axhline(BASELINE_METRICAS["MCC"], ls=":", color="#7F8C8D", lw=1.6,
                   label=f"baseline ({BASELINE_METRICAS['MCC']:.2f})")
    ax.set_xticks(x)
    ax.set_xticklabels(nomes, rotation=22, ha="right", fontsize=9)
    ax.set_ylabel("MCC")
    ax.set_title(f"Lote '{ESTUDO}' · {NOME_MODELO}", fontsize=13, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 🏆 9. Avaliação do Campeão

Curvas ROC e Precision-Recall, matriz de confusão em **contagens absolutas** e as
importâncias das variáveis.

> A matriz não é normalizada de propósito: o percentual esconde que a linha dos
> negativos tem pouquíssimos casos, e é justamente essa escassez que deveria moderar
> a leitura.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  AVALIAÇÃO DO CAMPEÃO
# ══════════════════════════════════════════════════════════════════════════════
_pipe = CAMPEAO["pipeline"]
_cols = CAMPEAO["features"]
_met = CAMPEAO["metricas_teste"]
_yproba, _ypred = _met["y_proba"], _met["y_pred"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
fig.suptitle(f"Campeão: {NOME_MODELO} · {CAMPEAO['variacao']} · split {SPLIT_HASH}",
             fontsize=14, fontweight="bold")

# (a) ROC
ax = axes[0]
fpr, tpr, _ = roc_curve(y_test, _yproba)
ax.plot(fpr, tpr, lw=2.4, color=COR_MODELO,
        label=f"AUC = {_met['AUC']:.3f}\nIC [{_met['AUC_ic_low']:.2f}–{_met['AUC_ic_high']:.2f}]")
ax.plot([0, 1], [0, 1], ls="--", color="#7F8C8D", lw=1.2, label="acaso")
ax.set_xlabel("Falso positivo (1 − specificity)")
ax.set_ylabel("Verdadeiro positivo (recall)")
ax.set_title("Curva ROC", fontsize=11, fontweight="bold")
ax.legend(fontsize=8.5, loc="lower right")
ax.grid(alpha=0.3)

# (b) Precision-Recall — mais informativa que ROC sob desbalanceamento
ax = axes[1]
prec, rec, _ = precision_recall_curve(y_test, _yproba)
ax.plot(rec, prec, lw=2.4, color=COR_MODELO, label=f"AP = {_met['AP']:.3f}")
_prev = float(y_test.mean())
ax.axhline(_prev, ls="--", color="#7F8C8D", lw=1.3,
           label=f"prevalência ({_prev:.2f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precisão")
ax.set_title("Curva Precisão-Recall (classe 1)", fontsize=11, fontweight="bold")
ax.legend(fontsize=8.5)
ax.grid(alpha=0.3)

# (c) Matriz de confusão em CONTAGENS ABSOLUTAS
ax = axes[2]
cm = confusion_matrix(y_test, _ypred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
            linewidths=1.5, linecolor="white",
            xticklabels=["Prev. bom", "Prev. risco"],
            yticklabels=["Real bom", "Real risco"])
ax.set_title(f"Matriz de confusão (threshold {_met['threshold']:.2f})",
             fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

# ── Leitura numérica ──────────────────────────────────────────────────────────
tn, fp, fn_, tp = cm.ravel()
print("=" * 78)
print(f"{'MÉTRICAS DO CAMPEÃO NO HOLDOUT':^78}")
print("=" * 78)
print(f"\n  {'Métrica':<26} {'Valor':>9}   {'IC 95%':>18}")
print("  " + "─" * 58)
print(f"  {'AUC':<26} {_met['AUC']:>9.4f}   "
      f"[{_met['AUC_ic_low']:.3f} – {_met['AUC_ic_high']:.3f}]")
print(f"  {'Recall (risco, classe 1)':<26} {_met['Recall']:>9.4f}   "
      f"[{_met['Recall_ic_low']:.3f} – {_met['Recall_ic_high']:.3f}]")
print(f"  {'Specificity (bom, classe 0)':<26} {_met['Specificity']:>9.4f}   "
      f"[{_met['Specificity_ic_low']:.3f} – {_met['Specificity_ic_high']:.3f}]")
print(f"  {'Balanced accuracy':<26} {_met['Balanced_Accuracy']:>9.4f}")
print(f"  {'MCC':<26} {_met['MCC']:>9.4f}")
print(f"  {'Brier (calibração)':<26} {_met['Brier']:>9.4f}   menor é melhor")
print("  " + "─" * 58)
print(f"\n  Matriz  ·  VN={tn}  FP={fp}  FN={fn_}  VP={tp}")
print(f"     FN = {fn_} cliente(s) de risco NÃO detectado(s) — o erro caro do negócio.")
print(f"     FP = {fp} bom(ns) pagador(es) marcado(s) como risco — atrito comercial.")

# ── Metas de negócio ──────────────────────────────────────────────────────────
print(f"\n{'─'*78}")
print(f"  METAS DE NEGÓCIO (declaradas antes do experimento)")
print(f"{'─'*78}")
for _rot, _chave, _meta in [("AUC-ROC", "AUC", METAS_KPI["auc_roc"]),
                            ("Recall classe 1", "Recall", METAS_KPI["recall_classe1"]),
                            ("Accuracy", "Accuracy", METAS_KPI["accuracy"])]:
    _v = _met[_chave]
    print(f"    {'✅' if _v >= _meta else '❌'} {_rot:<18} {_v:.4f}  (meta {_meta:.2f})")

if BASELINE_METRICAS is not None and _met["Accuracy"] <= BASELINE_METRICAS["Accuracy"]:
    print(f"\n  ⚠️  A accuracy não supera a do baseline ({BASELINE_METRICAS['Accuracy']:.4f}).")
    print(f"      Sob classe majoritária isso é comum e NÃO invalida o modelo — o que")
    print(f"      importa é o MCC e a balanced accuracy, que medem as duas classes.")

In [ ]:
# ─── Importância das variáveis ────────────────────────────────────────────────
def nomes_features(pipeline, features_num, features_cat) -> list:
    """Nomes das colunas APÓS o one-hot — sem isso os coeficientes não têm rótulo."""
    prep = pipeline.named_steps["prep"]
    try:
        cat_enc = prep.named_transformers_["cat"]
        cat_names = list(cat_enc.get_feature_names_out(features_cat))
    except Exception:
        cat_names = list(features_cat)
    return list(features_num) + cat_names


_fn = [c for c in _cols if c in FEATURES_NUM]
_fc = [c for c in _cols if c in FEATURES_CAT]
_nomes = nomes_features(_pipe, _fn, _fc)
_clf = _pipe.named_steps["clf"]

fig, ax = plt.subplots(figsize=(11, 6.5))

if hasattr(_clf, "feature_importances_"):
    imp = pd.DataFrame({"feature": _nomes, "valor": _clf.feature_importances_})
    imp = imp.sort_values("valor", ascending=False).head(15).sort_values("valor")
    ax.barh(imp["feature"], imp["valor"], color=COR_MODELO, alpha=0.9)
    ax.set_xlabel("Importância (redução de impureza)")
    _titulo = "Importância das variáveis"
    _nota = ("Importância por impureza favorece variáveis de alta cardinalidade. "
             "Leia como ordem de grandeza, não como ranking exato.")
elif hasattr(_clf, "coef_"):
    coefs = _clf.coef_[0]
    imp = pd.DataFrame({"feature": _nomes, "valor": coefs, "abs": np.abs(coefs)})
    imp = imp.sort_values("abs", ascending=False).head(15).sort_values("abs")
    cores = ["#E74C3C" if v > 0 else "#3498DB" for v in imp["valor"]]
    ax.barh(imp["feature"], imp["valor"], color=cores, alpha=0.9)
    ax.axvline(0, color="#2C3E50", lw=1)
    ax.set_xlabel("Coeficiente (log-odds)")
    _titulo = "Coeficientes  ·  vermelho aumenta o risco, azul reduz"
    _nota = ("Coeficientes são comparáveis entre si porque as numéricas foram "
             "padronizadas. Cada unidade é 1 desvio-padrão da variável.")
else:
    imp = None
    _titulo = "Modelo sem importâncias expostas"
    _nota = ""

ax.set_title(f"{_titulo}\n{NOME_MODELO} · {CAMPEAO['variacao']}",
             fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

if _nota:
    print(f"  ℹ️  {_nota}")

if imp is not None:
    print(f"\n  Top 8 variáveis:")
    for _, r in imp.sort_values("valor", key=abs, ascending=False).head(8).iterrows():
        print(f"     {r['feature']:<36} {r['valor']:>9.4f}")

    # As features derivadas do alvo dominarem o ranking é o sintoma direto do
    # vazamento parcial documentado — vale checar explicitamente.
    _suspeitas = [f for f in imp["feature"] if "ATRASO" in str(f).upper()]
    if _suspeitas:
        _topo = imp.sort_values("valor", key=abs, ascending=False).head(3)["feature"].tolist()
        if any(s in _topo for s in _suspeitas):
            print(f"\n  ⚠️  Uma feature derivada do alvo está no topo do ranking.")
            print(f"      É o vazamento parcial documentado em LIMITACOES_CONHECIDAS.")
            print(f"      Rode ESTUDO='ablacao' para medir quanto do desempenho vem daí.")

## 📦 10. Exportação do Campeão

O modelo é registrado no MLflow Model Registry com **a configuração inteira** anexada.
Um modelo sem a configuração que o produziu não é auditável: não se sabe qual limiar
aplicar, quais features esperar, nem qual regra de negócio ele reproduz.

> O registro no Model Registry é a cópia durável. O PKL local vive no disco efêmero do
> serverless e desaparece entre sessões.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  EXPORTAÇÃO DO CAMPEÃO
# ══════════════════════════════════════════════════════════════════════════════
METADADOS_MODELO = {
    "nome_modelo": NOME_MODELO,
    "algoritmo": CLASSE_MODELO.__name__,
    "variacao_campea": CAMPEAO["variacao"],
    "estudo": ESTUDO,
    "hiperparametros": CAMPEAO["hp"],
    "preproc": PREPROC,
    "features": CAMPEAO["features"],
    "features_num": _fn,
    "features_cat": _fc,
    "target": TARGET,
    # Contrato de inferência: sem isso, quem carregar o modelo não sabe com que
    # limiar ele foi avaliado nem como traduzir a probabilidade em faixa.
    "threshold_classificacao": THRESHOLD["classificacao"],
    "threshold_otimo_oof": CAMPEAO["threshold_otimo_oof"],
    "faixas_risco": {"baixo": THRESHOLD["faixa_baixo"], "medio": THRESHOLD["faixa_medio"]},
    # Proveniência
    "split_hash": SPLIT_HASH,
    "data_version": DATA_VERSION,
    "tabela_split": TABELA_SPLIT or CSV_SPLIT,
    "config_hash": CAMPEAO["config_hash"],
    "random_state": RANDOM_STATE,
    "run_id": CAMPEAO["run_id"],
    "run_pai_id": RUN_PAI_ID,
    "data_treinamento": datetime.now().isoformat(),
    "metricas_teste": {k: v for k, v in _met.items()
                       if isinstance(v, (int, float)) and not isinstance(v, bool)},
    "metricas_cv": CAMPEAO["metricas_cv"],
    "limitacoes": LIMITACOES_CONHECIDAS,
}

print("=" * 78)
print(f"{'📦 EXPORTAÇÃO DO CAMPEÃO':^78}")
print("=" * 78)

# ── Anexa os metadados ao run do campeão ──────────────────────────────────────
try:
    with mlflow.start_run(run_id=CAMPEAO["run_id"]):
        mlflow.log_dict(METADADOS_MODELO, "model_card.json")
        mlflow.set_tags({
            "campeao_do_lote": "true",
            "criterio_promocao": f"{SELECAO['metrica']} ({SELECAO['origem']})",
        })
    print(f"   ✅ model_card.json anexado ao run {CAMPEAO['run_id'][:12]}...")
except Exception as e:
    print(f"   ⚠️  Anexo indisponível ({type(e).__name__}): {e}")

# ── Registro no Model Registry ────────────────────────────────────────────────
# Unity Catalog usa nome de três níveis. É a cópia durável: o PKL local vive no
# disco efêmero do serverless e some entre sessões.
URI_MODELO = f"runs:/{CAMPEAO['run_id']}/modelo"
try:
    versao = mlflow.register_model(
        model_uri=URI_MODELO,
        name=NOME_REGISTRADO,
        tags={
            "estudo": ESTUDO,
            "variacao": CAMPEAO["variacao"],
            "split_hash": SPLIT_HASH,
            "data_version": DATA_VERSION,
            "modelo": NOME_MODELO,
            "mcc_teste": f"{_met['MCC']:.4f}",
        },
    )
    print(f"   ✅ Registrado: {NOME_REGISTRADO} versão {versao.version}")
except Exception as e:
    print(f"   ⚠️  Model Registry indisponível ({type(e).__name__}): {e}")
    print(f"      O modelo continua acessível por: {URI_MODELO}")

print(f"\n{'─'*78}")
print(f"  PARA CARREGAR EM PRODUÇÃO:")
print(f"{'─'*78}")
print(f'     modelo = mlflow.sklearn.load_model("{URI_MODELO}")')
print(f'     proba  = modelo.predict_proba(df[{CAMPEAO["features"][:2]} + ...])[:, 1]')
print(f'     risco  = "ALTO" if proba >= {THRESHOLD["faixa_medio"]} else \\')
print(f'              "MÉDIO" if proba >= {THRESHOLD["faixa_baixo"]} else "BAIXO"')
print(f"{'─'*78}")

In [ ]:
# ─── Resumo do lote ───────────────────────────────────────────────────────────
print("=" * 82)
print(f"{'🎉 LOTE CONCLUÍDO':^82}")
print("=" * 82)
print(f"  Modelo       : {NOME_MODELO}  ({CLASSE_MODELO.__name__})")
print(f"  Estudo       : {ESTUDO}")
print(f"  Nota         : {NOTA_ESTUDO}")
print("-" * 82)
print(f"  SPLIT_HASH   : {SPLIT_HASH}    ← use para comparar entre modelos")
print(f"  DATA_VERSION : {DATA_VERSION}")
print(f"  Run pai      : {RUN_PAI_ID}")
print("-" * 82)
print(f"  Variações executadas : {len(RESULTADOS)} de {len(LOTE)} planejadas")
print(f"  Campeã               : {CAMPEAO['variacao']}")
print(f"     ├─ MCC (teste)    : {_met['MCC']:.4f}")
print(f"     ├─ BalAcc (teste) : {_met['Balanced_Accuracy']:.4f}")
print(f"     ├─ AUC (teste)    : {_met['AUC']:.4f} "
      f"[{_met['AUC_ic_low']:.3f}–{_met['AUC_ic_high']:.3f}]")
print(f"     └─ Threshold OOF  : {CAMPEAO['threshold_otimo_oof']:.2f}")
print("=" * 82)
print()
print("  COMO NAVEGAR NO PAINEL DO MLFLOW")
print("  " + "─" * 78)
print(f"   · esta rodada        → expanda o run pai '{ESTUDO}' e use 'Compare'")
print(f"   · comparar ALGORITMOS→ filtre tags.split_hash = '{SPLIT_HASH}'")
print(f"                          e agrupe por tags.modelo")
print(f"   · comparar ESTUDOS   → filtre tags.estudo = '<nome>'")
print(f"   · efeito de um eixo  → filtre tags.eixo_varrido = '<parametro>'")
print(f"                          e ordene por metrics.test_mcc")
print("  " + "─" * 78)
print()
print("  PRÓXIMOS ESTUDOS SUGERIDOS PARA ESTE NOTEBOOK")
print("  " + "─" * 78)
print(f"   1. ESTUDO='ablacao'        → quanto do desempenho vem das features")
print(f"                                derivadas do alvo (o vazamento conhecido)")
print(f"   2. ESTUDO='elegibilidade'  → quantos clientes vale a pena usar")
print(f"   3. ESTUDO='janela_temporal'→ qual histórico generaliza melhor")
print(f"   Troque ESTUDO no painel, ajuste NOTA_ESTUDO e re-execute.")
print("=" * 82)